## KoBO Questionnaire Validator
Validates a KoBO (XLSForm) country questionnaire against a reference template.

Key differences from GeoPoll:
- Questions in **survey** sheet: `type`, `name`, `label::`, `relevant`, `required`, `constraint`, `Mandatory `
- Options in **choices** sheet: `list_name`, `name`, `label::`
- Skip logic = **relevant** column (XLSForm: `${var} = '1'`)
- Mandatory normalised from **Mandatory** column (yes / yes - panel / no)

In [90]:
import re
import polars as pl
import openpyxl
from pathlib import Path
from dataclasses import dataclass

## Step 1 -- Configuration
Set your file paths and language here.

In [91]:
import yaml

# -- Load centralized config ---------------------------------------------------
# Edit validation_config.yaml -- never touch this cell
_cfg_path = Path(__file__).parent / "validation_config.yaml" if "__file__" in dir() else \
            Path("validation_config.yaml")
cfg = yaml.safe_load(_cfg_path.read_text(encoding="utf-8"))["kobo"]

# Backward-compatible path resolution:
# - preferred: working_dir + questionnaire_file
# - fallback : questionnaire_path (legacy)
_working_dir = Path(cfg.get("working_dir") or cfg.get("output_dir") or "C:/Temp")
if cfg.get("questionnaire_file"):
    _q_path = _working_dir / str(cfg["questionnaire_file"])
elif cfg.get("questionnaire_path"):
    _q_path = Path(cfg["questionnaire_path"])
else:
    raise ValueError("Provide either 'questionnaire_file' or 'questionnaire_path' in validation_config.yaml")

cfg["working_dir"] = str(_working_dir)
cfg["questionnaire_path"] = str(_q_path)
if not cfg.get("output_dir"):
    cfg["output_dir"] = str(_working_dir)

print(f"Config loaded from: {_cfg_path}")
print(f"  Working  : {cfg['working_dir']}")
print(f"  Country  : {Path(cfg['questionnaire_path']).name}")
print(f"  Language : {cfg['language']}")
print(f"  ISO3     : {cfg['iso3']}")
print(f"  Mode     : {cfg['reference_mode']}")
print(f"  Output   : {cfg.get('output_dir', '(same as questionnaire)')}")


Config loaded from: validation_config.yaml
  Working  : C:\Temp
  Country  : DIEM_Monitoring questionnaire_kobo_en_AFG_20260415_test.xlsx
  Language : en
  ISO3     : AFG
  Mode     : previous_round
  Output   : C:/Temp


## Step 2 -- Reference file resolution

In [92]:
# Maps language code -> label column name in KoBO XLSForms
LANG_LABEL_COL = {
    "en": "label::English (en)",
    "fr": "label::French (fr)",
    "ar": "label::Arabic (ar)",
    "es": "label::Spanish (es)",
}

def find_kobo_template(language: str, templates_dir: Path) -> Path:
    lang_upper = language.upper()
    candidates = sorted(
        templates_dir.glob(f"*kobo*{lang_upper}*template*.xlsx"),
        key=lambda p: p.stat().st_mtime, reverse=True,
    )
    if not candidates:
        raise FileNotFoundError(f"No KoBO template for language {language!r} in {templates_dir}")
    return candidates[0]

TEMPLATES_DIR = Path(cfg["templates_dir"])
working_dir   = Path(cfg.get("working_dir") or cfg.get("output_dir") or Path(cfg["questionnaire_path"]).parent)
template_path = find_kobo_template(cfg["language"], TEMPLATES_DIR)

if cfg["reference_mode"] == "latest_template":
    ref_path = template_path
elif cfg["reference_mode"] == "previous_round":
    if cfg.get("previous_round_file"):
        ref_path = working_dir / str(cfg["previous_round_file"])
    elif cfg.get("previous_round_path"):
        ref_path = Path(cfg["previous_round_path"])
    elif cfg.get("custom_reference_path"):
        # legacy fallback
        ref_path = Path(cfg["custom_reference_path"])
    else:
        raise ValueError("reference_mode='previous_round' requires previous_round_file (or previous_round_path)")
else:
    raise ValueError(f"Unknown reference_mode: {cfg['reference_mode']!r}")

run = {
    "questionnaire_path": cfg["questionnaire_path"],
    "reference_path"    : str(ref_path),
    "template_path"     : str(template_path),
    "reference_mode"    : cfg["reference_mode"],
    "language"          : cfg["language"],
    "iso3"              : cfg["iso3"],
    "working_dir"       : str(working_dir),
    "output_dir"        : cfg.get("output_dir") or str(working_dir),
}

print(f"Reference : {Path(run['reference_path']).name}")
if run["reference_mode"] == "previous_round":
    print(f"Template  : {Path(run['template_path']).name}  (placeholder map only)")
print(f"Country   : {Path(run['questionnaire_path']).name}")
print(f"Language  : {run['language']}   ISO3: {run['iso3']}")


Reference : AFG validated_questionnaire_kobo_fr_AFG_20250120195700.xlsx
Template  : household_questionnaire_kobo_EN_template_20250708_ISO3_F2F.xlsx  (placeholder map only)
Country   : DIEM_Monitoring questionnaire_kobo_en_AFG_20260415_test.xlsx
Language  : en   ISO3: AFG


## Step 3 -- Readers
`read_kobo_survey` -- loads the **survey** sheet, normalises column names.  
`read_kobo_choices` -- loads the **choices** sheet.  
`build_question_options` -- joins them: one row per (Q Name, option).

In [93]:
_METADATA_TYPES = {
    "start", "end", "today", "deviceid", "phonenumber",
    "username", "simserial", "subscriberid", "background-audio",
}
_SELECT_TYPES = {"select_one", "select_multiple"}
_DATA_TYPES = {
    "select_one", "select_multiple", "text", "integer", "decimal",
    "geopoint", "geotrace", "geoshape", "date", "time", "datetime",
    "file", "image", "audio", "video", "barcode", "range", "calculate",
}

def _find_label_col(headers: list, language: str) -> str | None:
    target = LANG_LABEL_COL.get(language.lower(), f"label::{language}")
    if target in headers:
        return target
    return next((h for h in headers if h.startswith("label::")), None)


def _normalize_mandatory(val) -> str:
    v = str(val or "").strip().lower()
    if v in ("yes", "true", "mandatory"):
        return "mandatory"
    if v in ("yes - panel", "yes-panel", "mandatory-panel"):
        return "mandatory-panel"
    if v in ("no", "false"):
        return "non-mandatory"
    return ""


def read_kobo_survey(path: str, language: str = "en", _wb=None) -> pl.DataFrame:
    EMPTY = {
        "Q Name": pl.Utf8, "Q Type": pl.Utf8, "list_name": pl.Utf8,
        "label": pl.Utf8, "required": pl.Utf8, "mandatory_category": pl.Utf8,
        "relevant": pl.Utf8, "constraint": pl.Utf8,
        "appearance": pl.Utf8, "calculation": pl.Utf8, "hint": pl.Utf8,
        "excel_row": pl.Int64, "source_file": pl.Utf8,
    }
    owns_wb = _wb is None
    wb = _wb or openpyxl.load_workbook(path, data_only=True, read_only=True)
    try:
        ws   = wb["survey"]
        rows = list(ws.iter_rows(values_only=True))
    finally:
        if owns_wb:
            wb.close()

    if not rows:
        return pl.DataFrame(schema=EMPTY)

    headers   = [str(h).strip() if h is not None else "" for h in rows[0]]
    col_idx   = {h: i for i, h in enumerate(headers)}
    label_col = _find_label_col(headers, language)
    mand_col  = next((h for h in headers if h.strip() == "Mandatory"), None)

    def get(row, col):
        i = col_idx.get(col)
        return row[i] if i is not None and i < len(row) else None

    records = []
    for excel_row, row in enumerate(rows[1:], start=2):
        q_type_raw = str(get(row, "type") or "").strip()
        q_name     = str(get(row, "name") or "").strip()
        if not q_name or not q_type_raw or q_type_raw in _METADATA_TYPES:
            continue

        parts     = q_type_raw.split(None, 1)
        q_type    = parts[0]
        list_name = parts[1].strip() if len(parts) > 1 and q_type in _SELECT_TYPES else None

        # Support legacy KoBo encoding where list name is embedded with underscore,
        # e.g. "select_one_debt" / "select_multiple_debt".
        if list_name is None:
            if q_type_raw.startswith("select_one_"):
                q_type = "select_one"
                list_name = q_type_raw[len("select_one_"):].strip() or None
            elif q_type_raw.startswith("select_multiple_"):
                q_type = "select_multiple"
                list_name = q_type_raw[len("select_multiple_"):].strip() or None

        mand_cat = _normalize_mandatory(get(row, mand_col))
        if not mand_cat and q_type in _DATA_TYPES:
            mand_cat = "non-mandatory"
        if q_name.startswith("o_"):
            mand_cat = "optional"

        records.append({
            "Q Name"            : q_name,
            "Q Type"            : q_type,
            "list_name"         : list_name,
            "label"             : str(get(row, label_col) or "") if label_col else "",
            "required"          : str(get(row, "required")    or "").strip().lower(),
            "mandatory_category": mand_cat,
            "relevant"          : str(get(row, "relevant")    or "").strip(),
            "constraint"        : str(get(row, "constraint")  or "").strip(),
            "appearance"        : str(get(row, "appearance")  or "").strip(),
            "calculation"       : str(get(row, "calculation") or "").strip(),
            "hint"             : str(get(row, "hint")        or "").strip(),
            "excel_row"         : excel_row,
            "source_file"       : str(path),
        })
    return pl.DataFrame(records) if records else pl.DataFrame(schema=EMPTY)


def read_kobo_choices(path: str, language: str = "en", _wb=None) -> pl.DataFrame:
    EMPTY = {"list_name": pl.Utf8, "option_name": pl.Utf8, "option_label": pl.Utf8}
    owns_wb = _wb is None
    wb = _wb or openpyxl.load_workbook(path, data_only=True, read_only=True)
    try:
        ws   = wb["choices"]
        rows = list(ws.iter_rows(values_only=True))
    finally:
        if owns_wb:
            wb.close()
    if not rows:
        return pl.DataFrame(schema=EMPTY)

    headers   = [str(h).strip() if h is not None else "" for h in rows[0]]
    col_idx   = {h: i for i, h in enumerate(headers)}
    label_col = _find_label_col(headers, language)

    def get(row, col):
        i = col_idx.get(col)
        return row[i] if i is not None and i < len(row) else None

    records = []
    for row in rows[1:]:
        list_name = get(row, "list_name")
        name_raw  = get(row, "name")
        if not list_name or name_raw is None:
            continue
        records.append({
            "list_name"   : str(list_name).strip(),
            "option_name" : str(name_raw).strip(),
            "option_label": str(get(row, label_col) or "").strip() if label_col else "",
        })
    return pl.DataFrame(records) if records else pl.DataFrame(schema=EMPTY)


def build_question_options(survey_df: pl.DataFrame, choices_df: pl.DataFrame) -> pl.DataFrame:
    return (
        survey_df
        .filter(pl.col("Q Type").is_in(list(_SELECT_TYPES)))
        .filter(pl.col("list_name").is_not_null() & (pl.col("list_name") != ""))
        .select(["Q Name", "Q Type", "list_name", "source_file"])
        .join(choices_df, on="list_name", how="left")
        .select(["Q Name", "Q Type", "list_name", "option_name", "option_label", "source_file"])
    )


## Step 4 -- Load files
Open each workbook once, pass it to both readers, then close.

In [94]:
current_wb   = openpyxl.load_workbook(run["questionnaire_path"], data_only=True, read_only=True)
reference_wb = openpyxl.load_workbook(run["reference_path"],     data_only=True, read_only=True)

current_survey    = read_kobo_survey(run["questionnaire_path"],  run["language"], _wb=current_wb)
current_choices   = read_kobo_choices(run["questionnaire_path"], run["language"], _wb=current_wb)
current_wb.close()

reference_survey  = read_kobo_survey(run["reference_path"],  run["language"], _wb=reference_wb)
reference_choices = read_kobo_choices(run["reference_path"], run["language"], _wb=reference_wb)
reference_wb.close()

# Template is needed as placeholder map in previous_round mode.
if run.get("reference_mode") == "previous_round":
    template_wb = openpyxl.load_workbook(run["template_path"], data_only=True, read_only=True)
    template_survey  = read_kobo_survey(run["template_path"],  run["language"], _wb=template_wb)
    template_choices = read_kobo_choices(run["template_path"], run["language"], _wb=template_wb)
    template_wb.close()
else:
    template_survey  = reference_survey
    template_choices = reference_choices

current_options   = build_question_options(current_survey,   current_choices)
reference_options = build_question_options(reference_survey, reference_choices)
template_options  = build_question_options(template_survey,  template_choices)

print(f"Current  : {current_survey.height} questions, {current_choices.height} choice rows")
print(f"Reference: {reference_survey.height} questions, {reference_choices.height} choice rows")
if run.get("reference_mode") == "previous_round":
    print(f"Template : {template_survey.height} questions, {template_choices.height} choice rows")
print()
print(current_survey.select(["Q Name", "Q Type", "mandatory_category", "relevant"]).head(12))


Current  : 330 questions, 1653 choice rows
Reference: 240 questions, 1346 choice rows
Template : 221 questions, 516 choice rows

shape: (12, 4)
┌────────────────┬─────────────┬────────────────────┬─────────────────────┐
│ Q Name         ┆ Q Type      ┆ mandatory_category ┆ relevant            │
│ ---            ┆ ---         ┆ ---                ┆ ---                 │
│ str            ┆ str         ┆ str                ┆ str                 │
╞════════════════╪═════════════╪════════════════════╪═════════════════════╡
│ audit          ┆ audit       ┆                    ┆                     │
│ audio_random   ┆ calculate   ┆ non-mandatory      ┆                     │
│ enum_team      ┆ select_one  ┆ non-mandatory      ┆                     │
│ enumerator     ┆ select_one  ┆ mandatory          ┆                     │
│ enumerator_    ┆ calculate   ┆ mandatory          ┆                     │
│ …              ┆ …           ┆ …                  ┆ …                   │
│ resp_agree     ┆ s

## Step 4b -- Placeholder normalization

KoBO templates use `#placeholder#` tokens (e.g. `#ADMIN1#`, `#age#`, `#season#`) in label / constraint / option-label cells.
Country questionnaires have those tokens already replaced with real values.

**Before comparing**, we restore the country questionnaire's placeholder cells back to the template's vanilla form so the diff only flags genuine deviations -- not the expected country-specific substitutions.
**After comparing**, the report shows the original filled-in values (not the tokens) via the `current_orig` parameter passed to each comparison function.

In [95]:
_PLACEHOLDER_RE = re.compile(r'#[^#]+#')


def read_additional_info(country_path: str, language: str = "en") -> dict[str, str]:
    """
    Read the country questionnaire's Additional information sheet.
    Layout: row 1 empty, row 2 = headers (Original | Replacement), rows 3+ = data.
    Returns {#placeholder#: actual_value} for every filled-in entry.
    """
    try:
        wb = openpyxl.load_workbook(country_path, data_only=True, read_only=True)
        if "Additional information" not in wb.sheetnames:
            wb.close()
            return {}
        ws   = wb["Additional information"]
        rows = list(ws.iter_rows(values_only=True))
        wb.close()
    except Exception as e:
        print(f"Warning: could not read Additional information sheet: {e}")
        return {}

    if len(rows) < 2:
        return {}
    headers = [str(h).strip() if h is not None else "" for h in rows[1]]

    orig_idx = next((i for i, h in enumerate(headers) if h.lower().startswith("original")), None)
    if language == "fr":
        repl_idx = next(
            (i for i, h in enumerate(headers) if "replacement" in h.lower() and "fr" in h.lower()), None
        )
    elif language == "ar":
        repl_idx = next(
            (i for i, h in enumerate(headers) if "replacement" in h.lower() and "ar" in h.lower()), None
        )
    else:
        repl_idx = next((i for i, h in enumerate(headers) if h.lower() == "replacement"), None)
        if repl_idx is None:
            repl_idx = next((i for i, h in enumerate(headers) if "replacement" in h.lower()), None)

    if orig_idx is None or repl_idx is None:
        print("Warning: Original/Replacement columns not found in Additional information sheet.")
        return {}

    def _placeholder_aliases(s: str) -> set[str]:
        """Generate tolerant alias tokens (case + simple singular/plural) for replacements."""
        s = (s or "").strip()
        if not s:
            return set()
        aliases = {s, s.lower(), s.upper()}
        low = s.lower()
        # Simple singular/plural tolerance (e.g., unit <-> units)
        if low.endswith("s") and len(s) > 1:
            stem = s[:-1]
            aliases |= {stem, stem.lower(), stem.upper()}
        else:
            plus = s + "s"
            aliases |= {plus, plus.lower(), plus.upper()}
        return aliases

    pairs: dict[str, str] = {}
    for row in rows[2:]:
        orig = row[orig_idx] if orig_idx < len(row) else None
        repl = row[repl_idx] if repl_idx < len(row) else None
        if not orig or str(orig).strip() == "":
            continue
        repl_str = str(repl).strip() if repl is not None else ""
        if repl_str in ("", "nan", "None"):
            continue
        for _alias in _placeholder_aliases(str(orig)):
            pairs[f"#{_alias}#"] = repl_str
    return pairs


def restore_to_vanilla(
    current_df  : pl.DataFrame,
    reference_df: pl.DataFrame,
    cols        : list[str],
    key_cols    : list[str] | None = None,
) -> pl.DataFrame:
    """
    For every cell where the reference contains a #placeholder# token, overwrite
    the current questionnaire's value with the reference (vanilla) value.

    This prevents placeholder-filled country values from being flagged as deviations.
    key_cols: join keys -- defaults to ["Q Name"]; use ["Q Name", "option_name"] for options.
    cols    : columns to restore (e.g. ["label", "constraint"] or ["option_label"]).
    """
    if key_cols is None:
        key_cols = ["Q Name"]
    result = current_df
    for col in cols:
        if col not in reference_df.columns or col not in current_df.columns:
            continue
        ref_ph = (
            reference_df
            .filter(pl.col(col).str.contains(r'#[^#]+#'))
            .select(key_cols + [col])
            .rename({col: "__ref_vanilla__"})
        )
        if ref_ph.height == 0:
            continue
        result = (
            result
            .join(ref_ph, on=key_cols, how="left")
            .with_columns(
                pl.when(pl.col("__ref_vanilla__").is_not_null())
                .then(pl.col("__ref_vanilla__"))
                .otherwise(pl.col(col))
                .alias(col)
            )
            .drop("__ref_vanilla__")
        )
    return result


def apply_replacements(
    df   : pl.DataFrame,
    pairs: dict[str, str],
    cols : list[str],
) -> pl.DataFrame:
    """Apply #placeholder# -> actual_value replacements to specified columns (re-personalise after comparison)."""
    if not pairs:
        return df
    result = df
    for col in cols:
        if col not in result.columns:
            continue
        expr = pl.col(col)
        for key, val in sorted(pairs.items(), key=lambda kv: len(kv[0]), reverse=True):
            expr = expr.str.replace_all(key, val, literal=True)
        result = result.with_columns(expr.alias(col))
    return result


_CROP_SPECIALS_COMPARE = {
    "crop" : [("777", "No crop production"), ("888", "Don't know"), ("999", "Refused")],
    "crop2": [("666", "No other crop"),       ("888", "Don't know"), ("999", "Refused")],
    "crop3": [("666", "No other crop"),       ("888", "Don't know"), ("999", "Refused")],
}


def read_crop_choice_rows_for_compare(country_path: str, language: str) -> dict[str, list[tuple[str, str]]]:
    """Read current questionnaire Crop list and build rows for crop/crop2/crop3 replacement."""
    try:
        wb = openpyxl.load_workbook(country_path, data_only=True, read_only=True)
        if "Crop list" not in wb.sheetnames:
            wb.close()
            return {}
        rows = list(wb["Crop list"].iter_rows(values_only=True))
        wb.close()
    except Exception as e:
        print(f"Warning: could not read Crop list sheet: {e}")
        return {}

    if len(rows) < 3:
        return {}

    headers  = [str(h).strip() if h is not None else "" for h in rows[2]]
    sel_idx  = next((i for i, h in enumerate(headers) if "Select top" in h), None)
    code_idx = next((i for i, h in enumerate(headers) if "Dataset code" in h), None)
    if language == "fr":
        lbl_idx = next((i for i, h in enumerate(headers) if "Label" in h and "FR" in h), None)
    else:
        lbl_idx = next((i for i, h in enumerate(headers) if "Label" in h and "EN" in h), None)

    if code_idx is None or lbl_idx is None:
        return {}

    selected, unselected = [], []
    for row in rows[3:]:
        code = row[code_idx] if code_idx < len(row) else None
        lbl  = row[lbl_idx]  if lbl_idx  < len(row) else None
        sel  = row[sel_idx]  if sel_idx is not None and sel_idx < len(row) else None
        if code is None or str(code).strip() in ("", "nan"):
            continue
        entry = (str(code).strip(), str(lbl or "").strip())
        (selected if sel is not None and str(sel).strip() else unselected).append(entry)

    selected.sort(key=lambda x: x[0])
    unselected.sort(key=lambda x: x[0])
    combined = selected + unselected
    return {name: combined + specials for name, specials in _CROP_SPECIALS_COMPARE.items()}


def fetch_admin_choice_rows_for_compare(iso3: str) -> dict[str, list[tuple]]:
    """Fetch admin1/admin2 rows from AGOL for comparison preprocessing."""
    try:
        import urllib.request as _urlreq
        import json as _json

        _BASE = ("https://services5.arcgis.com/sjP4Ugu5s0dZWLjd/arcgis/rest/services"
                 "/Administrative_Boundaries_Reference_(view_layer)/FeatureServer")

        def _get(layer, fields):
            url = (f"{_BASE}/{layer}/query"
                   f"?where=adm0_ISO3+%3D+%27{iso3}%27"
                   f"&outFields={fields}&returnGeometry=false&outSR=4326&f=json")
            with _urlreq.urlopen(url, timeout=30) as r:
                return _json.loads(r.read().decode())["features"]

        adm1 = sorted(
            [(f["attributes"]["adm1_pcode"], f["attributes"]["adm1_name"])
             for f in _get(1, "adm1_name,adm1_pcode")],
            key=lambda x: x[0],
        )
        adm2 = sorted(
            [(f["attributes"]["adm2_pcode"], f["attributes"]["adm2_name"], f["attributes"]["adm1_pcode"])
             for f in _get(0, "adm2_name,adm2_pcode,adm1_pcode")],
            key=lambda x: x[0],
        )
        print(f"AGOL compare rows loaded: admin1={len(adm1)} admin2={len(adm2)}")
        return {"admin1": adm1, "admin2": adm2}
    except Exception as e:
        print(f"Warning: AGOL fetch failed during compare preprocessing: {e}")
        return {}


def replace_choice_lists_for_compare(choices_df: pl.DataFrame, country_rows: dict[str, list[tuple]]) -> pl.DataFrame:
    """Replace selected list_name blocks in choices_df with provided rows."""
    if not country_rows:
        return choices_df

    skip = list(country_rows.keys())
    kept = choices_df.filter(~pl.col("list_name").is_in(skip)) if choices_df.height > 0 else choices_df

    recs = []
    for list_name, entries in country_rows.items():
        for entry in entries:
            recs.append({
                "list_name": str(list_name),
                "option_name": str(entry[0]),
                "option_label": str(entry[1] if len(entry) > 1 else ""),
            })

    new_df = (
        pl.DataFrame(recs)
        if recs else
        pl.DataFrame(schema={"list_name": pl.Utf8, "option_name": pl.Utf8, "option_label": pl.Utf8})
    )

    if kept.height == 0:
        return new_df
    if new_df.height == 0:
        return kept
    return pl.concat([kept, new_df], how="vertical")


In [96]:
_VANILLA_COLS_SURVEY  = ["label", "constraint", "hint"]
_VANILLA_COLS_OPTIONS = ["option_label"]

# Read placeholder replacements from current questionnaire Additional information sheet
replacement_pairs = read_additional_info(run["questionnaire_path"], run["language"])
print(f"Loaded {len(replacement_pairs)} placeholder pair(s) from Additional information sheet:")
for k, v in replacement_pairs.items():
    print(f"  {k!r}  ->  {v!r}")

# Template is always the placeholder map for restoration in previous_round mode.
_vanilla_ref_survey  = template_survey
_vanilla_ref_options = template_options

# Restore: where template has #placeholder#, copy template value -> current
current_survey_vanilla  = restore_to_vanilla(
    current_survey,  _vanilla_ref_survey,  _VANILLA_COLS_SURVEY
)
current_options_vanilla = restore_to_vanilla(
    current_options, _vanilla_ref_options, _VANILLA_COLS_OPTIONS,
    key_cols=["Q Name", "option_name"],
)

_n_survey  = sum(
    _vanilla_ref_survey.filter(pl.col(c).str.contains(r'#[^#]+#')).height
    for c in _VANILLA_COLS_SURVEY if c in _vanilla_ref_survey.columns
)
_n_options = (
    _vanilla_ref_options.filter(pl.col("option_label").str.contains(r'#[^#]+#')).height
    if "option_label" in _vanilla_ref_options.columns else 0
)
print(f"\nVanilla restoration map (from template): {_n_survey} survey cell(s), {_n_options} option cell(s)")

# -----------------------------------------------------------------------------
# Comparison datasets
# - latest_template: current behaviour (vanilla compare)
# - previous_round : apply current round replacements BEFORE compare,
#                    including crop + AGOL list refresh for current only.
# -----------------------------------------------------------------------------
current_survey_cmp         = current_survey
current_survey_vanilla_cmp = current_survey_vanilla
current_options_vanilla_cmp = current_options_vanilla

_round_param_qnames = set()
_round_param_option_qnames = set()
_round_param_country_lists = set(cfg.get("country_specific_list_names", []))

# Placeholder-affected question names from template map.
for _c in _VANILLA_COLS_SURVEY:
    if _c in _vanilla_ref_survey.columns:
        _round_param_qnames |= set(
            _vanilla_ref_survey
            .filter(pl.col(_c).str.contains(r'#[^#]+#'))["Q Name"]
            .to_list()
        )
if "option_label" in _vanilla_ref_options.columns:
    _round_param_option_qnames |= set(
        _vanilla_ref_options
        .filter(pl.col("option_label").str.contains(r'#[^#]+#'))["Q Name"]
        .to_list()
    )

if run.get("reference_mode") == "previous_round":
    print("\nMode previous_round: preprocessing current questionnaire before comparison")

    # 1) Apply Additional information replacements to restored current values.
    current_survey_cmp = apply_replacements(current_survey_vanilla, replacement_pairs, _VANILLA_COLS_SURVEY)

    # 2) Rebuild current choices for country-specific lists from CURRENT round inputs.
    _crop_rows  = read_crop_choice_rows_for_compare(run["questionnaire_path"], run["language"])
    _admin_rows = fetch_admin_choice_rows_for_compare(run["iso3"])
    _country_rows = {**_crop_rows, **_admin_rows}

    current_choices_cmp = replace_choice_lists_for_compare(current_choices, _country_rows)
    current_options_cmp = build_question_options(current_survey_cmp, current_choices_cmp)

    # Restore+replace option labels using template placeholder map.
    current_options_cmp = restore_to_vanilla(
        current_options_cmp, _vanilla_ref_options, _VANILLA_COLS_OPTIONS,
        key_cols=["Q Name", "option_name"],
    )
    current_options_cmp = apply_replacements(current_options_cmp, replacement_pairs, _VANILLA_COLS_OPTIONS)

    current_survey_vanilla_cmp = current_survey_cmp
    current_options_vanilla_cmp = current_options_cmp

    # Country-specific option questions (crop/admin etc) are round-parameter-change candidates.
    if _round_param_country_lists:
        _country_qs = set(
            current_survey_cmp
            .filter(pl.col("list_name").is_in(list(_round_param_country_lists)))["Q Name"]
            .to_list()
        )
        _round_param_option_qnames |= _country_qs

    print(f"Current compare options rebuilt: {current_options_vanilla_cmp.height} rows")
else:
    print("\nMode latest_template: template-style comparison (no pre-replacement before compare)")

_round_param_qnames = sorted(_round_param_qnames)
_round_param_option_qnames = sorted(_round_param_option_qnames)


Loaded 78 placeholder pair(s) from Additional information sheet:
  '#PHONE NUMBERS#'  ->  '0700 700 7000'
  '#PHONE NUMBER#'  ->  '0700 700 7000'
  '#phone numbers#'  ->  '0700 700 7000'
  '#phone number#'  ->  '0700 700 7000'
  '#age#'  ->  '18'
  '#AGE#'  ->  '18'
  '#ages#'  ->  '18'
  '#AGES#'  ->  '18'
  '#admin1s#'  ->  'province'
  '#admin1#'  ->  'province'
  '#ADMIN1s#'  ->  'province'
  '#ADMIN1#'  ->  'province'
  '#ADMIN1S#'  ->  'province'
  '#admin2s#'  ->  'district'
  '#ADMIN2#'  ->  'district'
  '#ADMIN2s#'  ->  'district'
  '#admin2#'  ->  'district'
  '#ADMIN2S#'  ->  'district'
  '#reference years#'  ->  'last year'
  '#REFERENCE YEAR#'  ->  'last year'
  '#reference year#'  ->  'last year'
  '#REFERENCE YEARS#'  ->  'last year'
  '#season#'  ->  'last Season'
  '#SEASON#'  ->  'last Season'
  '#SEASONS#'  ->  'last Season'
  '#seasons#'  ->  'last Season'
  '#season phase#'  ->  'Harvesting'
  '#SEASON PHASE#'  ->  'Harvesting'
  '#SEASON PHASES#'  ->  'Harvesting'

## Step 5 -- Normalisation helpers

In [97]:
def normalize_text_expr(col_name: str) -> pl.Expr:
    return (
        pl.col(col_name).cast(pl.Utf8).fill_null("")
        .str.to_lowercase()
        .str.replace_all(r"[^\w\s]", "")
        .str.replace_all(r"\s+", " ")
        .str.strip_chars()
    )


def normalize_simple_expr(col_name: str) -> pl.Expr:
    return (
        pl.col(col_name).cast(pl.Utf8).fill_null("")
        .str.to_lowercase()
        .str.replace_all(r"\s+", " ")
        .str.strip_chars()
    )


def normalize_logic_expr(col_name: str) -> pl.Expr:
    return (
        pl.col(col_name).cast(pl.Utf8).fill_null("")
        .str.to_lowercase()
        .str.replace_all(r"\s+", " ")
        .str.replace_all(r"\s*([=<>!+\-*/(),])\s*", "$1")
        .str.strip_chars()
    )


## Step 6 -- Comparison functions

| Function | Checks | Adapted for KoBO? |
|---|---|---|
| `compare_question_presence` | Questions added/removed | No change |
| `compare_mandatory_kobo` | `mandatory_category` changed | Adapted (no raw yes/no) |
| `compare_option_labels_single` | Option label text changed | No change |
| `compare_option_presence_single` | Options added/removed | No change |
| `validate_relevant` | `relevant` broken refs + changes | KoBO-specific |

In [98]:
def compare_question_presence(current, reference):
    """Returns (added, removed). Each row has is_optional flag."""
    key = "Q Name"
    added = (
        current.select([key])
        .join(reference.select([key]), on=key, how="anti")
        .with_columns(pl.col(key).str.starts_with("o_").alias("is_optional"))
    )
    removed = (
        reference.select([key])
        .join(current.select([key]), on=key, how="anti")
        .with_columns(pl.col(key).str.starts_with("o_").alias("is_optional"))
    )
    return added, removed


def compare_mandatory_kobo(current, reference):
    """Compare mandatory_category between current and reference."""
    EMPTY = {
        "issue_type": pl.Utf8, "Q Name": pl.Utf8, "field": pl.Utf8,
        "Mandatory": pl.Utf8, "Mandatory_ref": pl.Utf8, "excel_row": pl.Int64,
    }
    if "mandatory_category" not in current.columns or "mandatory_category" not in reference.columns:
        return pl.DataFrame(schema=EMPTY)

    meaningful = {"mandatory", "mandatory-panel", "optional", "non-mandatory"}
    return (
        current.select(["Q Name", "mandatory_category", "excel_row"])
        .join(reference.select(["Q Name", "mandatory_category"]), on="Q Name", how="inner", suffix="_ref")
        .filter(
            pl.col("mandatory_category").is_in(list(meaningful)) |
            pl.col("mandatory_category_ref").is_in(list(meaningful))
        )
        .filter(pl.col("mandatory_category") != pl.col("mandatory_category_ref"))
        .with_columns([
            pl.lit("mandatory_column_mismatch").alias("issue_type"),
            pl.lit("mandatory_category").alias("field"),
        ])
        .rename({"mandatory_category": "Mandatory", "mandatory_category_ref": "Mandatory_ref"})
        .select(["issue_type", "Q Name", "field", "Mandatory", "Mandatory_ref", "excel_row"])
    )


def compare_list_name_changes(current, reference):
    """Flag questions where the choices list (list_name from type) changed."""
    EMPTY = {
        "issue_type": pl.Utf8, "set_name": pl.Utf8, "Q Name": pl.Utf8,
        "field": pl.Utf8, "current": pl.Utf8, "reference": pl.Utf8,
        "severity": pl.Utf8, "excel_row": pl.Int64,
    }
    cur = current.filter(pl.col("list_name").is_not_null()).select(["Q Name", "list_name", "excel_row"])
    ref = reference.filter(pl.col("list_name").is_not_null()).select(["Q Name", "list_name"])
    result = (
        cur.join(ref, on="Q Name", how="inner", suffix="_ref")
        .filter(pl.col("list_name") != pl.col("list_name_ref"))
        .with_columns([
            pl.lit("choices_list_changed").alias("issue_type"),
            pl.lit("").alias("set_name"),
            pl.lit("type (list_name)").alias("field"),
            pl.col("list_name").alias("current"),
            pl.col("list_name_ref").alias("reference"),
            pl.lit("medium").alias("severity"),
        ])
        .select(["issue_type", "set_name", "Q Name", "field", "current", "reference", "severity", "excel_row"])
    )
    return result if result.height > 0 else pl.DataFrame(schema=EMPTY)


def compare_question_labels(current_vanilla, reference, current_orig=None):
    """
    Compare question label text.
    current_vanilla: current survey with #placeholder# tokens restored (no false positives).
    current_orig   : original current survey -- used so the report shows the actual filled-in label.
    """
    EMPTY = {
        "issue_type": pl.Utf8, "set_name": pl.Utf8, "Q Name": pl.Utf8,
        "field": pl.Utf8, "current": pl.Utf8, "reference": pl.Utf8,
        "severity": pl.Utf8, "excel_row": pl.Int64,
    }
    if "label" not in current_vanilla.columns or "label" not in reference.columns:
        return pl.DataFrame(schema=EMPTY)

    diff = (
        current_vanilla.select(["Q Name", "label", "excel_row"])
        .join(reference.select(["Q Name", "label"]), on="Q Name", how="inner", suffix="_ref")
        .with_columns([
            normalize_text_expr("label").alias("_norm"),
            normalize_text_expr("label_ref").alias("_norm_ref"),
        ])
        .filter(pl.col("_norm") != pl.col("_norm_ref"))
        .filter(pl.col("label_ref") != "")
        .drop(["_norm", "_norm_ref"])
    )
    if diff.height == 0:
        return pl.DataFrame(schema=EMPTY)

    # Replace vanilla label with the actual filled-in label for display in the report
    if current_orig is not None and "label" in current_orig.columns:
        orig_labels = current_orig.select(["Q Name", "label"]).rename({"label": "_actual"})
        diff = (
            diff.join(orig_labels, on="Q Name", how="left")
            .with_columns(
                pl.when(pl.col("_actual").is_not_null())
                .then(pl.col("_actual"))
                .otherwise(pl.col("label"))
                .alias("label")
            )
            .drop("_actual")
        )

    return (
        diff
        .with_columns([
            pl.lit("label_mismatch").alias("issue_type"),
            pl.lit("").alias("set_name"),
            pl.lit("label").alias("field"),
            pl.col("label").alias("current"),
            pl.col("label_ref").alias("reference"),
            pl.lit("medium").alias("severity"),
        ])
        .select(["issue_type", "set_name", "Q Name", "field", "current", "reference", "severity", "excel_row"])
    )


def compare_constraint_changes(current_vanilla, reference, current_orig=None):
    """
    Compare constraint expressions.
    Uses vanilla current so #placeholder#-containing constraints are not flagged.
    """
    EMPTY = {
        "issue_type": pl.Utf8, "set_name": pl.Utf8, "Q Name": pl.Utf8,
        "field": pl.Utf8, "current": pl.Utf8, "reference": pl.Utf8,
        "severity": pl.Utf8, "excel_row": pl.Int64,
    }
    if "constraint" not in current_vanilla.columns or "constraint" not in reference.columns:
        return pl.DataFrame(schema=EMPTY)

    diff = (
        current_vanilla
        .select(["Q Name", "constraint", "excel_row"])
        .join(
            reference.select(["Q Name", "constraint"]),
            on="Q Name", how="inner", suffix="_ref",
        )
        .with_columns([
            normalize_logic_expr("constraint").alias("_norm"),
            normalize_logic_expr("constraint_ref").alias("_norm_ref"),
        ])
        .filter((pl.col("_norm") != pl.col("_norm_ref")) & ((pl.col("_norm") != "") | (pl.col("_norm_ref") != "")))
        .drop(["_norm", "_norm_ref"])
    )
    if diff.height == 0:
        return pl.DataFrame(schema=EMPTY)

    if current_orig is not None and "constraint" in current_orig.columns:
        orig_c = current_orig.select(["Q Name", "constraint"]).rename({"constraint": "_actual"})
        diff = (
            diff.join(orig_c, on="Q Name", how="left")
            .with_columns(
                pl.when(pl.col("_actual").is_not_null())
                .then(pl.col("_actual"))
                .otherwise(pl.col("constraint"))
                .alias("constraint")
            )
            .drop("_actual")
        )

    return (
        diff
        .with_columns([
            pl.lit("constraint_modified").alias("issue_type"),
            pl.lit("").alias("set_name"),
            pl.lit("constraint").alias("field"),
            pl.col("constraint").alias("current"),
            pl.col("constraint_ref").alias("reference"),
            pl.lit("medium").alias("severity"),
        ])
        .select(["issue_type", "set_name", "Q Name", "field", "current", "reference", "severity", "excel_row"])
    )


def compare_required_changes(current, reference):
    """Compare required column with whitespace/case-tolerant normalization."""
    EMPTY = {
        "issue_type": pl.Utf8, "set_name": pl.Utf8, "Q Name": pl.Utf8,
        "field": pl.Utf8, "current": pl.Utf8, "reference": pl.Utf8,
        "severity": pl.Utf8, "excel_row": pl.Int64,
    }
    if "required" not in current.columns or "required" not in reference.columns:
        return pl.DataFrame(schema=EMPTY)

    diff = (
        current.select(["Q Name", "required", "excel_row"])
        .join(reference.select(["Q Name", "required"]), on="Q Name", how="inner", suffix="_ref")
        .with_columns([
            normalize_simple_expr("required").alias("_norm"),
            normalize_simple_expr("required_ref").alias("_norm_ref"),
        ])
        .filter((pl.col("_norm") != pl.col("_norm_ref")) & ((pl.col("_norm") != "") | (pl.col("_norm_ref") != "")))
        .drop(["_norm", "_norm_ref"])
    )
    if diff.height == 0:
        return pl.DataFrame(schema=EMPTY)

    return (
        diff
        .with_columns([
            pl.lit("required_modified").alias("issue_type"),
            pl.lit("").alias("set_name"),
            pl.lit("required").alias("field"),
            pl.col("required").alias("current"),
            pl.col("required_ref").alias("reference"),
            pl.lit("medium").alias("severity"),
        ])
        .select(["issue_type", "set_name", "Q Name", "field", "current", "reference", "severity", "excel_row"])
    )


def compare_appearance_changes(current, reference):
    """Compare appearance with expression normalization that ignores tiny spacing differences."""
    EMPTY = {
        "issue_type": pl.Utf8, "set_name": pl.Utf8, "Q Name": pl.Utf8,
        "field": pl.Utf8, "current": pl.Utf8, "reference": pl.Utf8,
        "severity": pl.Utf8, "excel_row": pl.Int64,
    }
    if "appearance" not in current.columns or "appearance" not in reference.columns:
        return pl.DataFrame(schema=EMPTY)

    diff = (
        current.select(["Q Name", "appearance", "excel_row"])
        .join(reference.select(["Q Name", "appearance"]), on="Q Name", how="inner", suffix="_ref")
        .with_columns([
            normalize_logic_expr("appearance").alias("_norm"),
            normalize_logic_expr("appearance_ref").alias("_norm_ref"),
        ])
        .filter((pl.col("_norm") != pl.col("_norm_ref")) & ((pl.col("_norm") != "") | (pl.col("_norm_ref") != "")))
        .drop(["_norm", "_norm_ref"])
    )
    if diff.height == 0:
        return pl.DataFrame(schema=EMPTY)

    return (
        diff
        .with_columns([
            pl.lit("appearance_modified").alias("issue_type"),
            pl.lit("").alias("set_name"),
            pl.lit("appearance").alias("field"),
            pl.col("appearance").alias("current"),
            pl.col("appearance_ref").alias("reference"),
            pl.lit("medium").alias("severity"),
        ])
        .select(["issue_type", "set_name", "Q Name", "field", "current", "reference", "severity", "excel_row"])
    )


def compare_calculation_changes(current, reference):
    """Compare calculation with expression normalization that ignores tiny spacing differences."""
    EMPTY = {
        "issue_type": pl.Utf8, "set_name": pl.Utf8, "Q Name": pl.Utf8,
        "field": pl.Utf8, "current": pl.Utf8, "reference": pl.Utf8,
        "severity": pl.Utf8, "excel_row": pl.Int64,
    }
    if "calculation" not in current.columns or "calculation" not in reference.columns:
        return pl.DataFrame(schema=EMPTY)

    diff = (
        current.select(["Q Name", "calculation", "excel_row"])
        .join(reference.select(["Q Name", "calculation"]), on="Q Name", how="inner", suffix="_ref")
        .with_columns([
            normalize_logic_expr("calculation").alias("_norm"),
            normalize_logic_expr("calculation_ref").alias("_norm_ref"),
        ])
        .filter((pl.col("_norm") != pl.col("_norm_ref")) & ((pl.col("_norm") != "") | (pl.col("_norm_ref") != "")))
        .drop(["_norm", "_norm_ref"])
    )
    if diff.height == 0:
        return pl.DataFrame(schema=EMPTY)

    return (
        diff
        .with_columns([
            pl.lit("calculation_modified").alias("issue_type"),
            pl.lit("").alias("set_name"),
            pl.lit("calculation").alias("field"),
            pl.col("calculation").alias("current"),
            pl.col("calculation_ref").alias("reference"),
            pl.lit("medium").alias("severity"),
        ])
        .select(["issue_type", "set_name", "Q Name", "field", "current", "reference", "severity", "excel_row"])
    )



def compare_type_changes(current, reference):
    """Flag questions whose Q Type changed (e.g. select_one -> text)."""
    EMPTY = {
        "issue_type": pl.Utf8, "set_name": pl.Utf8, "Q Name": pl.Utf8,
        "field": pl.Utf8, "current": pl.Utf8, "reference": pl.Utf8,
        "severity": pl.Utf8, "excel_row": pl.Int64,
    }
    result = (
        current.select(["Q Name", "Q Type", "excel_row"])
        .join(reference.select(["Q Name", "Q Type"]), on="Q Name", how="inner", suffix="_ref")
        .filter(pl.col("Q Type") != pl.col("Q Type_ref"))
        .with_columns([
            pl.lit("type_changed").alias("issue_type"),
            pl.lit("").alias("set_name"),
            pl.lit("type").alias("field"),
            pl.col("Q Type").alias("current"),
            pl.col("Q Type_ref").alias("reference"),
            pl.lit("high").alias("severity"),
        ])
        .select(["issue_type", "set_name", "Q Name", "field", "current", "reference", "severity", "excel_row"])
    )
    return result if result.height > 0 else pl.DataFrame(schema=EMPTY)

def _canonical_option_key_expr(col_name: str) -> pl.Expr:
    """Canonical option key: strip spaces and treat integer-like decimals as same key (1 == 1.0)."""
    return (
        pl.col(col_name)
        .cast(pl.Utf8)
        .str.strip_chars()
        .str.replace(r"^([+-]?\d+)\.0+$", "$1")
    )


def compare_option_labels_single(current_opts, reference_opts, lang_scope="EN"):
    """Option label text changes for matching canonical Q Name + option_name."""
    cur = current_opts.with_columns([
        pl.col("Q Name").cast(pl.Utf8).str.strip_chars().str.to_lowercase().alias("_qk"),
        _canonical_option_key_expr("option_name").alias("_ok"),
    ])
    ref = reference_opts.with_columns([
        pl.col("Q Name").cast(pl.Utf8).str.strip_chars().str.to_lowercase().alias("_qk"),
        _canonical_option_key_expr("option_name").alias("_ok"),
    ])
    return (
        cur
        .join(ref, on=["_qk", "_ok"], how="inner", suffix="_ref")
        .with_columns([
            normalize_text_expr("option_label").alias("_norm"),
            normalize_text_expr("option_label_ref").alias("_norm_ref"),
        ])
        .filter(pl.col("_norm") != pl.col("_norm_ref"))
        .drop(["_norm", "_norm_ref", "_qk", "_ok"])
        .with_columns([
            pl.lit("option_label_mismatch").alias("issue_type"),
            pl.concat_str([pl.lit("option_"), pl.col("option_name")]).alias("field"),
            pl.lit("medium").alias("severity"),
            pl.lit(lang_scope).alias("lang_scope"),
            pl.lit(None).cast(pl.Int64).alias("excel_row"),
        ])
        .select(["issue_type", "Q Name", "field", "option_label", "option_label_ref",
                 "severity", "excel_row", "lang_scope"])
    )


def compare_option_presence_single(current_opts, reference_opts, lang_scope="EN"):
    """Added / removed options for matching canonical Q Name + option_name."""
    cur = current_opts.with_columns([
        pl.col("Q Name").cast(pl.Utf8).str.strip_chars().str.to_lowercase().alias("_qk"),
        _canonical_option_key_expr("option_name").alias("_ok"),
    ])
    ref = reference_opts.with_columns([
        pl.col("Q Name").cast(pl.Utf8).str.strip_chars().str.to_lowercase().alias("_qk"),
        _canonical_option_key_expr("option_name").alias("_ok"),
    ])
    key_cols = ["_qk", "_ok"]
    removed = (
        ref.select(key_cols + ["Q Name", "option_name", "option_label"])
        .join(cur.select(key_cols), on=key_cols, how="anti")
        .drop(["_qk", "_ok"])
        .with_columns([
            pl.lit("removed_option").alias("issue_type"),
            pl.concat_str([pl.lit("option_"), pl.col("option_name")]).alias("field"),
            pl.lit("high").alias("severity"),
            pl.lit(lang_scope).alias("lang_scope"),
        ])
        .select(["issue_type", "Q Name", "field", "option_label", "severity", "lang_scope"])
    )
    added = (
        cur.select(key_cols + ["Q Name", "option_name", "option_label"])
        .join(ref.select(key_cols), on=key_cols, how="anti")
        .drop(["_qk", "_ok"])
        .with_columns([
            pl.lit("added_option").alias("issue_type"),
            pl.concat_str([pl.lit("option_"), pl.col("option_name")]).alias("field"),
            pl.lit("medium").alias("severity"),
            pl.lit(lang_scope).alias("lang_scope"),
        ])
        .select(["issue_type", "Q Name", "field", "option_label", "severity", "lang_scope"])
    )
    return added, removed

def compare_hint_changes(current: pl.DataFrame, reference: pl.DataFrame) -> pl.DataFrame:
    """Flag questions where the hint text differs from the template (medium severity)."""
    EMPTY = {
        "issue_type": pl.Utf8, "set_name": pl.Utf8, "Q Name": pl.Utf8,
        "field": pl.Utf8, "current": pl.Utf8, "reference": pl.Utf8,
        "severity": pl.Utf8, "excel_row": pl.Int64,
    }
    ref_hints = reference.filter(pl.col("hint") != "").select(["Q Name", "hint"])
    if ref_hints.is_empty():
        return pl.DataFrame(schema=EMPTY)
    joined = (
        current.select(["Q Name", "hint", "excel_row"])
        .join(ref_hints, on="Q Name", how="inner", suffix="_ref")
        .filter(pl.col("hint") != pl.col("hint_ref"))
    )
    if joined.is_empty():
        return pl.DataFrame(schema=EMPTY)
    return (
        joined.with_columns([
            pl.lit("hint_changed").alias("issue_type"),
            pl.lit("").alias("set_name"),
            pl.col("hint").alias("current"),
            pl.col("hint_ref").alias("reference"),
            pl.lit("medium").alias("severity"),
            pl.lit("hint").alias("field"),
        ])
        .select(list(EMPTY.keys()))
    )


## Step 7 -- Relevant validation (KoBO skip logic)

Two layers:
1. **Broken reference** (high): `${var}` in `relevant` points to a Q Name not in the current survey
2. **Relevant modified** (medium): expression differs from template -- may need review

In [99]:
_KOBO_REF_RE = re.compile(r"\$\{([^}]+)\}")
_HASH_TOKEN_RE = re.compile(r"#([^#]+)#")
_LOOSE_DOLLAR_RE = re.compile(r"(?<!\{)\$[A-Za-z_][A-Za-z0-9_]*")


def validate_relevant(
    current_survey : pl.DataFrame,
    reference_survey: pl.DataFrame,
) -> pl.DataFrame:
    """
    Layer 1a - broken_relevant_reference (high): ${var} points to a name not in the survey at all
    Layer 1b - relevant_inexact_reference (high): exact name missing but o_<name> exists in survey
                (reference is broken in KoBO; note helps debugging)
    Layer 2  - relevant_modified (medium): expression changed from template
    """
    EMPTY = {
        "issue_type": pl.Utf8, "set_name": pl.Utf8, "Q Name": pl.Utf8,
        "field": pl.Utf8, "current": pl.Utf8, "reference": pl.Utf8,
        "severity": pl.Utf8, "excel_row": pl.Int64,
    }

    curr_qnames = set(current_survey["Q Name"].to_list())

    # Build lookup: stripped core name -> o_name for all optional-prefixed questions
    _optional_by_core: dict[str, str] = {
        qn[2:]: qn for qn in curr_qnames if qn.startswith("o_")
    }

    cur_rows = {r["Q Name"]: r for r in
                current_survey.select(["Q Name", "relevant", "excel_row"]).iter_rows(named=True)}
    ref_rows = {r["Q Name"]: r for r in
                reference_survey.select(["Q Name", "relevant"]).iter_rows(named=True)}

    issues = []
    for qname, cr in cur_rows.items():
        relevant  = str(cr.get("relevant") or "").strip()
        excel_row = cr.get("excel_row")

        refs = _KOBO_REF_RE.findall(relevant) if relevant else []
        broken = [v for v in refs if v not in curr_qnames]

        has_high_relevant_issue = False
        if broken:
            truly_missing = [v for v in broken if v not in _optional_by_core]
            has_similar   = [v for v in broken if v in _optional_by_core]

            if truly_missing:
                has_high_relevant_issue = True
                issues.append({
                    "issue_type": "broken_relevant_reference",
                    "set_name": "", "Q Name": qname, "field": "relevant",
                    "current"  : relevant[:220],
                    "reference": f"missing variables: {truly_missing}",
                    "severity" : "high", "excel_row": excel_row,
                })

            for v in has_similar:
                has_high_relevant_issue = True
                similar = _optional_by_core[v]
                issues.append({
                    "issue_type": "relevant_inexact_reference",
                    "set_name": "", "Q Name": qname, "field": "relevant",
                    "current"  : relevant[:220],
                    "reference": f"${{{v}}} not found; {similar!r} exists with similar name",
                    "severity" : "high", "excel_row": excel_row,
                })

        # Layer 2: changed from template (only when no high relevant integrity issue).
        # Apply only when question exists in BOTH files.
        # (Added questions should be checked structurally, not compared as template drift.)
        ref_relevant = str(ref_rows.get(qname, {}).get("relevant") or "").strip()
        if (not has_high_relevant_issue) and (qname in ref_rows) and (ref_relevant != relevant):
            issues.append({
                "issue_type": "relevant_modified",
                "set_name": "", "Q Name": qname, "field": "relevant",
                "current"  : relevant[:220],
                "reference": ref_relevant[:220],
                "severity" : "medium", "excel_row": excel_row,
            })

    if not issues:
        return pl.DataFrame(schema=EMPTY)
    return pl.DataFrame(issues).unique(
        subset=["issue_type", "Q Name", "field", "current", "reference"], keep="first"
    )


def validate_questionnaire_structure(
    current_survey: pl.DataFrame,
    template_survey: pl.DataFrame | None = None,
    replacement_pairs: dict[str, str] | None = None,
) -> pl.DataFrame:
    """
    Structural checks for token/placeholder integrity in survey text/formula fields.
    - #token# unresolved against template placeholders + Additional information
    - #token# that looks like a survey variable (likely ${var} expected)
    - loose $var syntax (likely ${var} expected)
    - ${var} missing from survey (outside relevant column checks)
    """
    EMPTY = {
        "issue_type": pl.Utf8, "set_name": pl.Utf8, "Q Name": pl.Utf8,
        "field": pl.Utf8, "current": pl.Utf8, "reference": pl.Utf8,
        "severity": pl.Utf8, "excel_row": pl.Int64,
    }
    _fields = [c for c in ["label", "hint", "constraint", "calculation"] if c in current_survey.columns]
    if not _fields:
        return pl.DataFrame(schema=EMPTY)

    curr_qnames = set(current_survey["Q Name"].to_list())

    # Tokens considered resolvable: known in template OR in Additional information pairs.
    known_tokens = set()
    if template_survey is not None and template_survey.height > 0:
        tpl_fields = [c for c in _fields if c in template_survey.columns]
        if tpl_fields:
            for row in template_survey.select(tpl_fields).iter_rows(named=True):
                for f in tpl_fields:
                    txt = str(row.get(f) or "")
                    for tok in _HASH_TOKEN_RE.findall(txt):
                        known_tokens.add(f"#{tok}#")
    if replacement_pairs:
        known_tokens |= set(replacement_pairs.keys())
    known_tokens_l = {t.lower() for t in known_tokens}

    issues = []
    iter_cols = ["Q Name", "excel_row"] + _fields
    for row in current_survey.select(iter_cols).iter_rows(named=True):
        qname = row.get("Q Name", "")
        excel_row = row.get("excel_row")
        for field in _fields:
            txt = str(row.get(field) or "").strip()
            if not txt:
                continue

            # 1) #token# checks
            for tok in _HASH_TOKEN_RE.findall(txt):
                full = f"#{tok}#"
                tok_core = str(tok).strip()

                if tok_core in curr_qnames:
                    issues.append({
                        "issue_type": "placeholder_should_use_kobo_ref",
                        "set_name": "questionnaire_structure",
                        "Q Name": qname,
                        "field": field,
                        "current": txt[:220],
                        "reference": f"{full} matches survey variable '{tok_core}' - use ${{{tok_core}}}",
                        "severity": "high",
                        "excel_row": excel_row,
                    })
                elif full.lower() not in known_tokens_l:
                    issues.append({
                        "issue_type": "placeholder_not_found",
                        "set_name": "questionnaire_structure",
                        "Q Name": qname,
                        "field": field,
                        "current": txt[:220],
                        "reference": f"{full} not in template placeholders or Additional information",
                        "severity": "high",
                        "excel_row": excel_row,
                    })

            # 2) $var (missing braces) checks
            for m in _LOOSE_DOLLAR_RE.findall(txt):
                var = m[1:]
                if var in curr_qnames:
                    ref = f"{m} uses loose syntax; use ${{{var}}}"
                    sev = "high"
                else:
                    ref = f"{m} uses loose syntax; expected ${{var}} format"
                    sev = "medium"
                issues.append({
                    "issue_type": "kobo_ref_loose_syntax",
                    "set_name": "questionnaire_structure",
                    "Q Name": qname,
                    "field": field,
                    "current": txt[:220],
                    "reference": ref,
                    "severity": sev,
                    "excel_row": excel_row,
                })

            # 3) ${var} checks (outside relevant field)
            for v in _KOBO_REF_RE.findall(txt):
                if v not in curr_qnames:
                    issues.append({
                        "issue_type": "kobo_ref_missing_variable",
                        "set_name": "questionnaire_structure",
                        "Q Name": qname,
                        "field": field,
                        "current": txt[:220],
                        "reference": f"${{{v}}} not found in survey Q Name",
                        "severity": "high",
                        "excel_row": excel_row,
                    })

    if not issues:
        return pl.DataFrame(schema=EMPTY)
    return pl.DataFrame(issues).unique(
        subset=["issue_type", "Q Name", "field", "current", "reference"], keep="first"
    )


## Step 8 -- Critical sets
Same logic as GeoPoll. Load `critical_sets.yaml` if present; otherwise skip.
Note: `mandatory_category` is passed directly -- no extra normalisation needed.

In [100]:
import yaml
from pathlib import Path as _Path

_CRIT_YAML = _Path(
    "c:/Users/edoar/WORK/FAO/repo/questionnaire_validation_revision/scripts/critical_sets.yaml"
)
rules = {"exact_sets": {}, "min_count_sets": {}, "crop_harvest": {}}
if _CRIT_YAML.exists():
    with open(_CRIT_YAML, encoding="utf-8") as _f:
        rules = yaml.safe_load(_f) or rules
    print(f"Loaded critical_sets.yaml  exact_sets: {len(rules.get('exact_sets', {}))}")
else:
    print("No critical_sets.yaml found -- critical-set checks will be skipped.")


def validate_critical_sets(questions_df, exact_sets):
    EMPTY = {
        "issue_type": pl.Utf8, "set_name": pl.Utf8, "Q Name": pl.Utf8,
        "field": pl.Utf8, "current": pl.Utf8, "reference": pl.Utf8,
        "severity": pl.Utf8, "excel_row": pl.Int64,
    }
    if not exact_sets:
        return pl.DataFrame(schema=EMPTY)

    # KoBO uses mandatory_category (already normalised); add Mandatory alias for compat
    if "Mandatory" not in questions_df.columns and "mandatory_category" in questions_df.columns:
        questions_df = questions_df.with_columns(
            pl.col("mandatory_category").alias("Mandatory")
        )
    # Mandatory_norm: map to yes/no for comparison with yaml expected values
    questions_df = questions_df.with_columns(
        pl.when(pl.col("Mandatory").is_in(["mandatory", "yes"])).then(pl.lit("yes"))
        .when(pl.col("Mandatory").is_in(["mandatory-panel"])).then(pl.lit("yes"))
        .when(pl.col("Mandatory").is_in(["optional", "no"])).then(pl.lit("no"))
        .otherwise(pl.lit("")).alias("Mandatory_norm")
    )

    present_qnames = set(questions_df["Q Name"].to_list())
    issues = []
    for set_name, set_rules in exact_sets.items():
        required_names   = [r["q_name"] for r in set_rules if r.get("required", True)]
        present_in_set   = [r["q_name"] for r in set_rules if r["q_name"] in present_qnames]
        present_required = [q for q in required_names if q in present_qnames]

        if 0 < len(present_required) < len(required_names):
            issues.append({
                "issue_type": "partial_critical_set", "set_name": set_name, "Q Name": "",
                "field": "Q Name",
                "current"  : ", ".join(sorted(present_in_set)),
                "reference": f"Expected all {len(required_names)} required questions",
                "severity" : "high", "excel_row": None,
            })

        for rule in set_rules:
            q_name   = rule["q_name"]
            expected = rule.get("expected_mandatory", "")
            required = rule.get("required", True)
            if q_name not in present_qnames:
                issues.append({
                    "issue_type": "missing_critical_question" if required else "advisory_question",
                    "set_name": set_name, "Q Name": q_name,
                    "field": "Q Name", "current": "",
                    "reference": "present",
                    "severity": "high" if required else "medium", "excel_row": None,
                })
                continue
            if not expected:
                continue
            row = questions_df.filter(pl.col("Q Name") == q_name).select(
                ["Q Name", "Mandatory", "Mandatory_norm", "excel_row"]
            ).to_dicts()
            if not row:
                continue
            row = row[0]
            if row["Mandatory_norm"] != expected:
                issues.append({
                    "issue_type": "critical_mandatory_mismatch", "set_name": set_name,
                    "Q Name": q_name, "field": "Mandatory",
                    "current": row["Mandatory"], "reference": expected,
                    "severity": "high", "excel_row": row.get("excel_row"),
                })

    if not issues:
        return pl.DataFrame(schema=EMPTY)
    return pl.DataFrame(issues)


def validate_prefix_counts(questions_df, min_count_sets):
    EMPTY = {
        "issue_type": pl.Utf8, "set_name": pl.Utf8, "Q Name": pl.Utf8,
        "field": pl.Utf8, "current": pl.Utf8, "reference": pl.Utf8,
        "severity": pl.Utf8, "excel_row": pl.Int64,
    }
    if not min_count_sets:
        return pl.DataFrame(schema=EMPTY)
    all_qnames = questions_df["Q Name"].to_list()
    issues = []
    for set_name, rule in min_count_sets.items():
        prefix    = rule.get("prefix", "")
        min_count = rule.get("min_count", 1)
        desc      = rule.get("description", f"At least {min_count} '{prefix}*' questions required")
        matched   = sorted(q for q in all_qnames if q.startswith(prefix))
        if len(matched) < min_count:
            found_str = f"{len(matched)} found" + (f": {', '.join(matched)}" if matched else " (none)")
            issues.append({
                "issue_type": "below_minimum_count", "set_name": set_name, "Q Name": "",
                "field": "count", "current": found_str, "reference": desc,
                "severity": "high", "excel_row": None,
            })
    if not issues:
        return pl.DataFrame(schema=EMPTY)
    return pl.DataFrame(issues)


def validate_crop_harvest(survey: pl.DataFrame, crop_harvest_cfg: dict) -> pl.DataFrame:
    """
    PASS if only the minimal set is present (no extra full-set questions), OR
    if the full set is completely present (extra questions beyond full are fine).
    FAIL for any partial overlap.
    Returns a DataFrame with ISSUE_SCHEMA columns.
    """
    EMPTY = {
        "issue_type": pl.Utf8, "set_name": pl.Utf8, "Q Name": pl.Utf8,
        "field": pl.Utf8, "current": pl.Utf8, "reference": pl.Utf8,
        "severity": pl.Utf8, "excel_row": pl.Int64,
    }
    if not crop_harvest_cfg:
        return pl.DataFrame(schema=EMPTY)

    minimal = set(crop_harvest_cfg.get("minimal", []))
    full    = set(crop_harvest_cfg.get("full", []))
    present = set(survey["Q Name"].to_list())

    if full.issubset(present):
        return pl.DataFrame(schema=EMPTY)

    extra_full = (full - minimal) & present
    if (minimal & present) == minimal and not extra_full:
        return pl.DataFrame(schema=EMPTY)

    missing_from_full = full - present
    partial_found     = (full - minimal) & present
    detail_parts = []
    if missing_from_full:
        detail_parts.append(f"missing from full set: {sorted(missing_from_full)}")
    if partial_found:
        detail_parts.append(f"partial full-set Qs present: {sorted(partial_found)}")
    detail = "; ".join(detail_parts) or "partial crop-harvest set"
    ref    = f"minimal={sorted(minimal)}, full={sorted(full)}"

    return pl.DataFrame([{
        "issue_type": "crop_harvest_violation",
        "set_name":   "CRP_HARV",
        "Q Name":     "crp_harv_*",
        "field":      "",
        "current":    detail,
        "reference":  ref,
        "severity":   "high",
        "excel_row":  -1,
    }], schema=EMPTY)

Loaded critical_sets.yaml  exact_sets: 1


## Step 9 -- Issue unifiers
Convert each comparison result to the common schema.

In [101]:
def make_presence_issues(added, removed, reference_survey=None):
    added_base = added

    # Build optional counterpart lookup from added questions: core -> o_<core>
    # Example: removed "confl_yn" + added "o_confl_yn" => mandatory_to_optional
    added_opt_lookup = (
        added
        .with_columns([
            pl.col("Q Name").cast(pl.Utf8).str.strip_chars().alias("_q"),
            pl.when(pl.col("Q Name").cast(pl.Utf8).str.strip_chars().str.starts_with("o_"))
            .then(pl.col("Q Name").cast(pl.Utf8).str.strip_chars().str.slice(2))
            .otherwise(pl.col("Q Name").cast(pl.Utf8).str.strip_chars())
            .str.to_lowercase()
            .alias("_core"),
        ])
        .filter(pl.col("_q").str.starts_with("o_"))
        .select(["_core", pl.col("Q Name").alias("_optional_qname")])
        .unique(subset=["_core"], keep="first")
    )

    # Severity: mandatory / mandatory-panel -> high; non-mandatory / optional -> info
    if reference_survey is not None and "mandatory_category" in reference_survey.columns:
        removed_aug = (
            removed.join(
                reference_survey.select(["Q Name", "mandatory_category"]),
                on="Q Name", how="left"
            )
            .with_columns(
                pl.when(pl.col("Q Name").cast(pl.Utf8).str.strip_chars().str.starts_with("o_"))
                .then(pl.col("Q Name").cast(pl.Utf8).str.strip_chars().str.slice(2))
                .otherwise(pl.col("Q Name").cast(pl.Utf8).str.strip_chars())
                .str.to_lowercase()
                .alias("_core")
            )
        )
        severity_expr = (
            pl.when(pl.col("mandatory_category").is_in(["mandatory", "mandatory-panel"]))
            .then(pl.lit("high"))
            .otherwise(pl.lit("info"))
            .alias("severity")
        )

        moved_to_optional = (
            removed_aug
            .join(added_opt_lookup, on="_core", how="inner")
            .filter(pl.col("mandatory_category").is_in(["mandatory", "mandatory-panel"]))
        )
    else:
        removed_aug = removed.with_columns([
            pl.lit("").alias("mandatory_category"),
            pl.when(pl.col("Q Name").cast(pl.Utf8).str.strip_chars().str.starts_with("o_"))
            .then(pl.col("Q Name").cast(pl.Utf8).str.strip_chars().str.slice(2))
            .otherwise(pl.col("Q Name").cast(pl.Utf8).str.strip_chars())
            .str.to_lowercase()
            .alias("_core"),
        ])
        severity_expr = (
            pl.when(pl.col("is_optional")).then(pl.lit("info")).otherwise(pl.lit("high"))
            .alias("severity")
        )
        moved_to_optional = pl.DataFrame(schema={
            "Q Name": pl.Utf8, "is_optional": pl.Boolean, "mandatory_category": pl.Utf8,
            "_core": pl.Utf8, "_optional_qname": pl.Utf8,
        })

    moved_i = (
        moved_to_optional.with_columns([
            pl.lit("mandatory_to_optional").alias("issue_type"),
            pl.lit("").alias("set_name"),
            pl.lit("Q Name").alias("field"),
            pl.col("_optional_qname").alias("current"),
            pl.lit("present (mandatory in reference)").alias("reference"),
            pl.lit("high").alias("severity"),
            pl.lit(None).cast(pl.Int64).alias("excel_row"),
        ])
        .select(["issue_type", "set_name", "Q Name", "field", "current", "reference", "severity", "excel_row"])
    )

    added_base = (
        added.join(moved_to_optional.select(pl.col("_optional_qname").alias("Q Name")), on="Q Name", how="anti")
        if moved_to_optional.height > 0 else added
    )
    added_i = added_base.with_columns([
        pl.lit("added_question").alias("issue_type"),
        pl.lit("").alias("set_name"),
        pl.lit("Q Name").alias("field"),
        pl.lit("present").alias("current"),
        pl.lit("missing_in_reference").alias("reference"),
        pl.lit("info").alias("severity"),
        pl.lit(None).cast(pl.Int64).alias("excel_row"),
    ]).select(["issue_type", "set_name", "Q Name", "field", "current", "reference", "severity", "excel_row"])

    removed_base = (
        removed_aug.join(moved_to_optional.select(["Q Name"]), on="Q Name", how="anti")
        if moved_to_optional.height > 0 else removed_aug
    )

    removed_i = (
        removed_base.with_columns([
            pl.lit("removed_question").alias("issue_type"),
            pl.lit("").alias("set_name"),
            pl.lit("Q Name").alias("field"),
            pl.lit("missing_in_current").alias("current"),
            pl.lit("present").alias("reference"),
            severity_expr,
            pl.lit(None).cast(pl.Int64).alias("excel_row"),
        ])
        .select(["issue_type", "set_name", "Q Name", "field", "current", "reference", "severity", "excel_row"])
    )

    return pl.concat([added_i, removed_i, moved_i], how="vertical")


def make_mandatory_issues(mandatory_diff):
    return (
        mandatory_diff
        .with_columns([
            pl.lit("").alias("set_name"),
            pl.col("Mandatory").alias("current"),
            pl.col("Mandatory_ref").alias("reference"),
            pl.lit("high").alias("severity"),
        ])
        .select(["issue_type", "set_name", "Q Name", "field", "current", "reference", "severity", "excel_row"])
    )


def make_option_issues(option_diff):
    return (
        option_diff
        .with_columns([
            pl.lit("").alias("set_name"),
            pl.col("option_label").alias("current"),
            pl.col("option_label_ref").alias("reference"),
            pl.lit("medium").alias("severity"),
        ])
        .select(["issue_type", "set_name", "Q Name", "field", "current", "reference", "severity", "excel_row"])
    )


def make_option_presence_issues(added_opts, removed_opts):
    added_i = added_opts.with_columns([
        pl.lit("").alias("set_name"),
        pl.col("option_label").alias("current"),
        pl.lit("(not in template)").alias("reference"),
        pl.lit("medium").alias("severity"),
        pl.lit(None).cast(pl.Int64).alias("excel_row"),
    ]).select(["issue_type", "set_name", "Q Name", "field", "current", "reference", "severity", "excel_row"])

    removed_i = removed_opts.with_columns([
        pl.lit("").alias("set_name"),
        pl.lit("(removed)").alias("current"),
        pl.col("option_label").alias("reference"),
        pl.lit("high").alias("severity"),
        pl.lit(None).cast(pl.Int64).alias("excel_row"),
    ]).select(["issue_type", "set_name", "Q Name", "field", "current", "reference", "severity", "excel_row"])

    return pl.concat([added_i, removed_i], how="vertical")


## Step 10 -- Run pipeline

In [102]:
import time
_t0 = time.perf_counter()

added, removed = compare_question_presence(current_survey_cmp, reference_survey)
print(f"Questions added: {added.height}  removed: {removed.height}")

mandatory_diff = compare_mandatory_kobo(current_survey_cmp, reference_survey)
print(f"Mandatory mismatches: {mandatory_diff.height}")

list_name_issues = compare_list_name_changes(current_survey_cmp, reference_survey)
print(f"Choices-list changes: {list_name_issues.height}")

# Field-level comparisons -- vanilla versions for label/constraint
label_issues       = compare_question_labels(current_survey_vanilla_cmp, reference_survey, current_survey_cmp)
constraint_issues  = compare_constraint_changes(current_survey_vanilla_cmp, reference_survey, current_survey_cmp)
type_issues        = compare_type_changes(current_survey_cmp, reference_survey)
required_issues    = compare_required_changes(current_survey_cmp, reference_survey)
appearance_issues  = compare_appearance_changes(current_survey_cmp, reference_survey)
calculation_issues = compare_calculation_changes(current_survey_cmp, reference_survey)
hint_issues        = compare_hint_changes(current_survey_cmp, reference_survey)
print(f"Label: {label_issues.height}  Constraint: {constraint_issues.height}  Type: {type_issues.height}")
print(f"Required: {required_issues.height}  Appearance: {appearance_issues.height}  Calculation: {calculation_issues.height}  Hint: {hint_issues.height}")

# Options -- in previous_round mode, include country-specific lists so round updates are visible
_country_lists  = set(cfg.get("country_specific_list_names", []))
if run.get("reference_mode") == "previous_round":
    _country_lists = set()

_list_changed_q = set(list_name_issues["Q Name"].to_list())

cur_opts_cmp = (
    current_options_vanilla_cmp
    .filter(~pl.col("list_name").is_in(list(_country_lists)))
    .filter(~pl.col("Q Name").is_in(list(_list_changed_q)))
)
ref_opts_cmp = (
    reference_options
    .filter(~pl.col("list_name").is_in(list(_country_lists)))
    .filter(~pl.col("Q Name").is_in(list(_list_changed_q)))
)

option_diff              = compare_option_labels_single(cur_opts_cmp, ref_opts_cmp)
added_opts, removed_opts = compare_option_presence_single(cur_opts_cmp, ref_opts_cmp)
print(f"Option label changes: {option_diff.height}  added: {added_opts.height}  removed: {removed_opts.height}")

relevant_issues = validate_relevant(current_survey_cmp, reference_survey)
print(f"Relevant issues: {relevant_issues.height}")

structure_issues = validate_questionnaire_structure(
    current_survey_cmp,
    template_survey=template_survey,
    replacement_pairs=replacement_pairs,
)
print(f"Questionnaire structure issues: {structure_issues.height}")

critical_issues = validate_critical_sets(current_survey_cmp, rules.get("exact_sets", {}))
count_issues    = validate_prefix_counts(current_survey_cmp, rules.get("min_count_sets", {}))
harvest_issues  = validate_crop_harvest(current_survey_cmp, rules.get("crop_harvest", {}))
print(f"Critical set issues: {critical_issues.height}  Count issues: {count_issues.height}")

# -- Assemble all issues -------------------------------------------------------
presence_issues     = make_presence_issues(added, removed, reference_survey)
mandatory_issues    = make_mandatory_issues(mandatory_diff)
option_label_issues = make_option_issues(option_diff)
option_pres_issues  = make_option_presence_issues(added_opts, removed_opts)

# Exclude option issues for questions not present in both files
_excluded = set(added["Q Name"].to_list()) | set(removed["Q Name"].to_list())
if _excluded:
    option_label_issues = option_label_issues.filter(~pl.col("Q Name").is_in(list(_excluded)))
    option_pres_issues  = option_pres_issues.filter(~pl.col("Q Name").is_in(list(_excluded)))

# Config-driven exclusions for operational fields (case/space-insensitive)
_cfg_excluded = {
    str(q).strip().lower()
    for q in cfg.get("exclude_qnames_from_option_compare", [])
    if q is not None and str(q).strip()
}
if _cfg_excluded:
    _q_norm = pl.col("Q Name").cast(pl.Utf8).str.strip_chars().str.to_lowercase()
    option_label_issues = option_label_issues.filter(~_q_norm.is_in(list(_cfg_excluded)))
    option_pres_issues  = option_pres_issues.filter(~_q_norm.is_in(list(_cfg_excluded)))

all_issues = pl.concat(
    [presence_issues, mandatory_issues, list_name_issues,
     label_issues, constraint_issues, type_issues,
     required_issues, appearance_issues, calculation_issues,
     hint_issues,
     option_label_issues, option_pres_issues,
     critical_issues, count_issues, harvest_issues, relevant_issues, structure_issues],
    how="vertical",
)

# -- previous_round special remap ---------------------------------------------
# Differences affecting replacement-driven cells/lists are tracked as
# round_parameter_change (info) instead of hard errors.
if run.get("reference_mode") == "previous_round":
    _rp_types_q = [
        "label_mismatch", "constraint_modified", "hint_changed", "choices_list_changed",
    ]
    _rp_types_opt = ["option_label_mismatch", "added_option", "removed_option"]
    _rp_qnames = sorted(set(_round_param_qnames) | set(_round_param_option_qnames))
    if _rp_qnames:
        _rp_mask_q = (
            pl.col("issue_type").is_in(_rp_types_q)
            & pl.col("Q Name").is_in(_rp_qnames)
        )
        _rp_mask_opt = (
            pl.col("issue_type").is_in(_rp_types_opt)
            & pl.col("Q Name").is_in(_rp_qnames)
        )
        all_issues = all_issues.with_columns([
            pl.when(_rp_mask_q).then(pl.lit("round_parameter_change")).otherwise(pl.col("issue_type")).alias("issue_type"),
            pl.when(_rp_mask_q | _rp_mask_opt).then(pl.lit("info")).otherwise(pl.col("severity")).alias("severity"),
        ])
        print(f"Round-parameter remap applied on {_rp_qnames.__len__()} question(s)")

# -- CS downgrade: removed questions in passing prefix-count sets -> info ------
_passing_prefixes = []
for _set_name, _rule in rules.get("min_count_sets", {}).items():
    if count_issues.filter(pl.col("set_name") == _set_name).height == 0:
        _prefix = _rule.get("prefix", "")
        if _prefix:
            _passing_prefixes.append(_prefix)

if _passing_prefixes:
    _exprs    = [pl.col("Q Name").cast(pl.Utf8).str.starts_with(p) for p in _passing_prefixes]
    _q_matches = pl.any_horizontal(_exprs) if len(_exprs) > 1 else _exprs[0]
    all_issues = all_issues.with_columns(
        pl.when((pl.col("issue_type") == "removed_question") & _q_matches)
        .then(pl.lit("info"))
        .otherwise(pl.col("severity"))
        .alias("severity")
    )

# -- Attach mandatory_cat -----------------------------------------------------
_mand_lookup = (
    pl.concat([
        reference_survey.select(["Q Name", "mandatory_category"]).with_columns(pl.lit(0).alias("_p")),
        current_survey_cmp.select(["Q Name", "mandatory_category"]).with_columns(pl.lit(1).alias("_p")),
    ], how="vertical")
    .sort(["Q Name", "_p"])
    .group_by("Q Name")
    .agg(pl.col("mandatory_category").last().alias("mandatory_cat"))
)
all_issues = (
    all_issues
    .join(_mand_lookup, on="Q Name", how="left")
    .with_columns(pl.col("mandatory_cat").fill_null(""))
)

# -- Final sort: high -> medium -> info (AFTER cs-downgrade + mandatory_cat) --
all_issues = (
    all_issues
    .with_columns(
        pl.when(pl.col("severity") == "high").then(pl.lit(0))
        .when(pl.col("severity") == "medium").then(pl.lit(1))
        .otherwise(pl.lit(2))
        .alias("_s")
    )
    .sort(["_s", "issue_type", "Q Name"])
    .drop("_s")
)

# -- Found info for Critical Sets ---------------------------------------------
_found_info = {}
_cur_qnames = current_survey_cmp["Q Name"].to_list()

# Exact sets: track which expected questions are present (used for summary fallback).
for _set_name, _set_rules in rules.get("exact_sets", {}).items():
    _expected = [r.get("q_name", "") for r in _set_rules]
    _found_info[_set_name] = [q for q in _expected if q in _cur_qnames]

# Prefix-count sets: keep matched names for display + count details.
for _set_name, _rule in rules.get("min_count_sets", {}).items():
    _prefix  = _rule.get("prefix", "")
    _matched = sorted(q for q in _cur_qnames if q.startswith(_prefix))
    _found_info[_set_name] = _matched

print(f"\nTotal issues: {all_issues.height}  ({time.perf_counter()-_t0:.2f}s)")
print(all_issues.select(["issue_type", "severity", "Q Name"]).head(15))




Questions added: 154  removed: 64
Mandatory mismatches: 0
Choices-list changes: 0
Label: 16  Constraint: 3  Type: 0
Required: 5  Appearance: 0  Calculation: 6  Hint: 0
Option label changes: 23  added: 929  removed: 220
Relevant issues: 35
Questionnaire structure issues: 3
Critical set issues: 2  Count issues: 0
Round-parameter remap applied on 29 question(s)

Total issues: 307  (0.04s)
shape: (15, 3)
┌────────────────────────────┬──────────┬───────────────────────────┐
│ issue_type                 ┆ severity ┆ Q Name                    │
│ ---                        ┆ ---      ┆ ---                       │
│ str                        ┆ str      ┆ str                       │
╞════════════════════════════╪══════════╪═══════════════════════════╡
│ broken_relevant_reference  ┆ high     ┆ generalInfo               │
│ broken_relevant_reference  ┆ high     ┆ ls_salesdif_              │
│ kobo_ref_missing_variable  ┆ high     ┆ crpmain                   │
│ kobo_ref_missing_variable  ┆ high 

## Step 11 -- Export

| Sheet | Content |
|---|---|
| **Summary** | Issue counts by category |
| **Critical Sets** | Structural validation (mandatory presence, count rules) |
| **Relevant Changes** | `relevant` broken references and modifications |
| **Question Changes** | Presence, mandatory flag, label text, constraint, choices list |
| **Option Changes** | Answer-option additions, removals, label changes |

In [103]:
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

import difflib
import re

try:
    from openpyxl.cell.rich_text import CellRichText, TextBlock
    from openpyxl.cell.text import InlineFont
    _RICH_TEXT_AVAILABLE = True
except Exception:
    CellRichText = None
    TextBlock = None
    InlineFont = None
    _RICH_TEXT_AVAILABLE = False

FILL_HIGH   = PatternFill("solid", fgColor="F4CCCC")
FILL_MEDIUM = PatternFill("solid", fgColor="FCE5CD")
FILL_INFO   = PatternFill("solid", fgColor="CFE2F3")
FILL_PASS   = PatternFill("solid", fgColor="D9EAD3")
FILL_HEADER = PatternFill("solid", fgColor="274E13")
FILL_SECT   = PatternFill("solid", fgColor="E8F5E9")

FONT_TITLE  = Font(bold=True, size=13, color="274E13")
FONT_HDR    = Font(bold=True, color="FFFFFF", size=11)
FONT_SECT   = Font(bold=True, color="274E13", size=11)
FONT_NORM   = Font(size=10)
THIN_BORDER = Border(
    left=Side(style="thin", color="CCCCCC"),
    right=Side(style="thin", color="CCCCCC"),
    bottom=Side(style="thin", color="CCCCCC"),
)
SEV_FILL = {"high": FILL_HIGH, "medium": FILL_MEDIUM, "info": FILL_INFO, "pass": FILL_PASS}

_ISSUE_LABELS = {
    "removed_question"         : "Question removed",
    "mandatory_to_optional"    : "Mandatory question moved to optional",
    "added_question"           : "Question added",
    "mandatory_column_mismatch": "Mandatory flag changed",
    "type_changed"             : "Question type changed",
    "label_mismatch"           : "Label text changed",
    "required_changed"         : "Required changed",
    "appearance_changed"       : "Appearance changed",
    "calculation_changed"      : "Calculation changed",
    "constraint_modified"      : "Constraint changed",
    "hint_changed"             : "Hint text changed",
    "crop_harvest_violation"   : "Crop harvest rule violation",
    "choices_list_changed"     : "Choices list changed",
    "round_parameter_change"  : "Round parameter change (Additional info / crop / AGOL)",
    "removed_option"           : "Option removed",
    "added_option"             : "Option added",
    "option_label_mismatch"    : "Option label changed",
    "placeholder_not_found"    : "Placeholder not in template/additional info",
    "placeholder_should_use_kobo_ref": "Placeholder looks like variable (use ${var})",
    "kobo_ref_loose_syntax"    : "Loose $var syntax (use ${var})",
    "kobo_ref_missing_variable": "${var} references missing variable",
}

def _hdr(ws, row, vals):
    for c, v in enumerate(vals, 1):
        cell = ws.cell(row=row, column=c, value=v)
        cell.fill = FILL_HEADER; cell.font = FONT_HDR
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.row_dimensions[row].height = 18

def _sect(ws, row, title, n):
    ws.merge_cells(start_row=row, start_column=1, end_row=row, end_column=n)
    c = ws.cell(row=row, column=1, value=title)
    c.fill = FILL_SECT; c.font = FONT_SECT
    c.alignment = Alignment(horizontal="left", vertical="center")
    ws.row_dimensions[row].height = 20

def _style_row(ws, row, n, severity=""):
    fill = SEV_FILL.get(severity)
    for c in range(1, n + 1):
        cell = ws.cell(row=row, column=c)
        cell.font = FONT_NORM; cell.border = THIN_BORDER
        cell.alignment = Alignment(vertical="top", wrap_text=True)
        if fill: cell.fill = fill

def _autofit(ws, mn=12, mx=60):
    for col_cells in ws.columns:
        w = max((len(str(c.value)) if c.value else 0) for c in col_cells)
        ws.column_dimensions[get_column_letter(col_cells[0].column)].width = min(max(w+2, mn), mx)

COL_MAP = [
    ("issue_type",    "Issue type"),
    ("mandatory_cat", "Type"),
    ("set_name",      "Set"),
    ("Q Name",        "Q Name"),
    ("field",         "Field"),
    ("current",       "Current value"),
    ("reference",     "Reference / rule"),
    ("severity",      "Severity"),
    ("excel_row",     "Excel row"),
]

def _table(ws, start_row, df, apply_view=True, col_map=None):
    cmap = col_map or COL_MAP
    cols = [(s, d) for s, d in cmap if s in df.columns]
    _hdr(ws, start_row, [d for _, d in cols])
    r = start_row + 1
    if df.height == 0:
        ws.cell(row=r, column=1, value="No issues in this category").font = Font(bold=True, color="274E13", size=10)
        return r + 2
    for rd in df.to_dicts():
        for c, (src, _) in enumerate(cols, 1):
            ws.cell(row=r, column=c, value=rd.get(src))
        _style_row(ws, r, len(cols), rd.get("severity", ""))
        r += 1
    if apply_view:
        ws.freeze_panes = f"A{start_row+1}"
        ws.auto_filter.ref = f"A{start_row}:{get_column_letter(len(cols))}{start_row}"
    return r + 1


def _tokenize_for_word_diff(text: str):
    # Tokenize into non-space chunks and attach following spaces to the same token.
    # This reduces rich-text run fragmentation and avoids visual space-loss artifacts.
    raw = re.findall(r"\s+|\S+", text)
    out = []
    for tok in raw:
        if tok.isspace() and out:
            out[-1] = out[-1] + tok
        else:
            out.append(tok)
    return out


def _coalesce_segments(parts):
    merged = []
    for txt, is_equal in parts:
        if not txt:
            continue
        if merged and merged[-1][1] == is_equal:
            merged[-1] = (merged[-1][0] + txt, is_equal)
        else:
            merged.append((txt, is_equal))
    return merged


def _build_diff_segments(current_text: str, reference_text: str):
    a = _tokenize_for_word_diff(current_text)
    b = _tokenize_for_word_diff(reference_text)
    sm = difflib.SequenceMatcher(a=a, b=b)
    current_parts, reference_parts = [], []
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag == "equal":
            current_parts.append(("".join(a[i1:i2]), True))
            reference_parts.append(("".join(b[j1:j2]), True))
        elif tag == "replace":
            current_parts.append(("".join(a[i1:i2]), False))
            reference_parts.append(("".join(b[j1:j2]), False))
        elif tag == "delete":
            current_parts.append(("".join(a[i1:i2]), False))
        elif tag == "insert":
            reference_parts.append(("".join(b[j1:j2]), False))
    return _coalesce_segments(current_parts), _coalesce_segments(reference_parts)


def _set_rich_text(cell, full_text: str, parts):
    if not _RICH_TEXT_AVAILABLE:
        cell.value = full_text
        return
    blocks = []
    for txt, is_equal in parts:
        if not txt:
            continue
        color = "FF000000" if is_equal else "FFFF0000"
        blocks.append(TextBlock(InlineFont(color=color), txt))
    if not blocks:
        cell.value = full_text
        return
    cell.value = CellRichText(*blocks)


def _apply_inline_diff_for_issue(ws, start_row, df, cols, issue_types):
    if df.height == 0:
        return
    cidx = {src: i for i, (src, _) in enumerate(cols, start=1)}
    issue_col = cidx.get("issue_type")
    cur_col = cidx.get("current")
    ref_col = cidx.get("reference")
    if not issue_col or not cur_col or not ref_col:
        return

    for offset, rd in enumerate(df.to_dicts(), start=1):
        if rd.get("issue_type") not in set(issue_types or []):
            continue
        cur_txt = str(rd.get("current") or "")
        ref_txt = str(rd.get("reference") or "")
        cur_parts, ref_parts = _build_diff_segments(cur_txt, ref_txt)
        _set_rich_text(ws.cell(row=start_row + offset, column=cur_col), cur_txt, cur_parts)
        _set_rich_text(ws.cell(row=start_row + offset, column=ref_col), ref_txt, ref_parts)


CRITICAL_ISSUE_TYPES = {
    "missing_critical_question", "advisory_question", "partial_critical_set",
    "critical_mandatory_mismatch", "below_minimum_count", "crop_harvest_violation",
}
RELEVANT_ISSUE_TYPES  = {"broken_relevant_reference", "relevant_inexact_reference", "relevant_modified"}
STRUCTURE_ISSUE_TYPES = {
    "placeholder_not_found", "placeholder_should_use_kobo_ref",
    "kobo_ref_loose_syntax", "kobo_ref_missing_variable",
}
QUESTION_CHANGE_TYPES = {
    "mandatory_column_mismatch", "added_question", "removed_question", "mandatory_to_optional",
    "choices_list_changed", "label_mismatch", "constraint_modified",
    "type_changed", "required_changed", "appearance_changed", "calculation_changed",
    "hint_changed",
    "round_parameter_change",
}


def _grouped_table(ws, r, df):
    """Render (issue_type, mandatory_cat, severity) -> count grouped summary table."""
    _hdr(ws, r, ["Issue type", "Q type", "Count", "Severity"]); r += 1
    if df.height == 0:
        ws.cell(row=r, column=1, value="No issues").font = Font(bold=True, color="274E13", size=10)
        return r + 2
    grouped = (
        df
        .group_by(["issue_type", "mandatory_cat", "severity"])
        .agg(pl.len().alias("count"))
        .with_columns(
            pl.when(pl.col("severity") == "high").then(pl.lit(0))
            .when(pl.col("severity") == "medium").then(pl.lit(1))
            .otherwise(pl.lit(2))
            .alias("_s")
        )
        .sort(["_s", "issue_type", "mandatory_cat"])
        .drop("_s")
    )
    for rd in grouped.to_dicts():
        ws.cell(row=r, column=1, value=_ISSUE_LABELS.get(rd["issue_type"], rd["issue_type"]))
        ws.cell(row=r, column=2, value=rd["mandatory_cat"] or "")
        ws.cell(row=r, column=3, value=rd["count"])
        ws.cell(row=r, column=4, value=rd["severity"])
        _style_row(ws, r, 4, rd["severity"])
        r += 1
    return r + 1


def write_summary_sheet(wb, all_issues, rules=None, critical_issues=None,
                        count_issues=None, found_info=None, replacement_status=None):
    ws = wb.create_sheet("Summary")
    ws.sheet_view.showGridLines = False
    for col, w in zip("ABCD", [38, 20, 42, 12]):
        ws.column_dimensions[col].width = w

    # Title
    ws.merge_cells("A1:D1")
    ws["A1"] = "KoBO Questionnaire Validation Report"
    ws["A1"].font = FONT_TITLE
    ws["A1"].alignment = Alignment(horizontal="left", vertical="center")
    ws.row_dimensions[1].height = 26

    r = 3
    _rules = rules or {}
    _crit  = (critical_issues if critical_issues is not None
              else pl.DataFrame(schema={"set_name": pl.Utf8, "issue_type": pl.Utf8, "Q Name": pl.Utf8}))
    _cnt   = (count_issues if count_issues is not None
              else pl.DataFrame(schema={"set_name": pl.Utf8, "issue_type": pl.Utf8}))
    _found = found_info or {}

    # -- Critical Sets Status -------------------------------------------------
    _sect(ws, r, "CRITICAL SETS STATUS", 4); r += 1
    _hdr(ws, r, ["Critical set", "Status", "Details", ""]); r += 1

    def _set_row(ws, r, name, passed, details):
        ws.cell(row=r, column=1, value=name)
        ws.cell(row=r, column=2, value="PASS" if passed else "FAIL")
        ws.cell(row=r, column=3, value=details)
        _style_row(ws, r, 4, "pass" if passed else "high")
        ws.cell(row=r, column=2).font = Font(bold=True, size=10,
                                              color="274E13" if passed else "CC0000")

    _EXACT_FAIL_TYPES = ["missing_critical_question", "partial_critical_set",
                          "critical_mandatory_mismatch"]
    for set_name, set_rules in _rules.get("exact_sets", {}).items():
        required_qs = [r.get("q_name", "") for r in set_rules if r.get("required", True)]
        advisory_qs = [r.get("q_name", "") for r in set_rules if not r.get("required", True)]
        found_exact = set(_found.get(set_name, []))
        missing_required = [q for q in required_qs if q and q not in found_exact]

        set_fail = all_issues.filter(
            (pl.col("set_name") == set_name) & pl.col("issue_type").is_in(_EXACT_FAIL_TYPES)
        )
        set_warn = all_issues.filter(
            (pl.col("set_name") == set_name) & (pl.col("issue_type") == "advisory_question")
        )

        # Fallback to rule-based presence check so status stays correct
        # even if issue tagging is affected by execution order.
        passed = (set_fail.height == 0) and (len(missing_required) == 0)

        if passed and set_warn.height == 0:
            details = "All required questions present"
        elif passed:
            warn_qs = [q for q in advisory_qs if q and q not in found_exact]
            if not warn_qs:
                warn_qs = [q for q in set_warn["Q Name"].to_list() if q]
            details = (f"Required OK  (advisory missing: {', '.join(warn_qs)})"
                       if warn_qs else "All required questions present")
        else:
            missing = list(dict.fromkeys(missing_required + [q for q in set_fail["Q Name"].to_list() if q]))
            details = f"Missing: {', '.join(missing)}" if missing else "Issues found"
        _set_row(ws, r, set_name, passed, details); r += 1
    for set_name, rule in _rules.get("min_count_sets", {}).items():
        min_count = rule.get("min_count", 1)
        prefix    = rule.get("prefix", "")
        found_qs  = _found.get(set_name, [])
        failed    = _cnt.filter(pl.col("set_name") == set_name).height > 0
        if not failed:
            details = f">={min_count} {prefix}* confirmed  ({len(found_qs)} found)"
        else:
            details = f"{len(found_qs)} found -- need >={min_count} {prefix}* questions"
        _set_row(ws, r, set_name, not failed, details); r += 1

    # Crop harvest row
    if _rules.get("crop_harvest"):
        _harv_fail = all_issues.filter(pl.col("issue_type") == "crop_harvest_violation").height > 0
        _harv_det = ("Partial crop-harvest set -- see issues" if _harv_fail
                     else "Crop harvest rule satisfied")
        _set_row(ws, r, "CRP_HARV", not _harv_fail, _harv_det); r += 1

    # Skip logic summary row
    _broken   = all_issues.filter(pl.col("issue_type").is_in(["broken_relevant_reference", "relevant_inexact_reference"])).height
    _modified = all_issues.filter(pl.col("issue_type") == "relevant_modified").height
    skip_ok = _broken == 0
    skip_det = (f"{_broken} broken reference(s); {_modified} modified" if not skip_ok
                else f"No broken references  ({_modified} modified)")
    _set_row(ws, r, "SKIP LOGIC (relevant)", skip_ok, skip_det); r += 1

    if not _rules.get("exact_sets") and not _rules.get("min_count_sets"):
        ws.cell(row=r, column=1, value="No critical sets defined")
        _style_row(ws, r, 4, "info"); r += 1

    r += 1

    # -- Question Changes -----------------------------------------------------
    _sect(ws, r, "QUESTION CHANGES", 4); r += 1
    r = _grouped_table(ws, r,
                       all_issues.filter(pl.col("issue_type").is_in(list(QUESTION_CHANGE_TYPES))))
    r += 1

    # -- Option Changes -------------------------------------------------------
    _sect(ws, r, "OPTION CHANGES  (questions present in both files)", 4); r += 1
    r = _grouped_table(ws, r,
                       all_issues.filter(pl.col("issue_type").is_in(
                           ["removed_option", "added_option", "option_label_mismatch"])))

    r += 1

    # -- Replacement Status ---------------------------------------------------
    _sect(ws, r, "REPLACEMENT STATUS  (validated questionnaire output)", 4); r += 1
    _hdr(ws, r, ["Check", "Status", "Details", ""]); r += 1

    _repl_rows = (replacement_status or {}).get("rows", [])
    if not _repl_rows:
        ws.cell(row=r, column=1, value="Validated questionnaire step not run yet")
        ws.cell(row=r, column=2, value="INFO")
        ws.cell(row=r, column=3, value="Run Step 12, then rerun export to include replacement checks.")
        _style_row(ws, r, 4, "info"); r += 1
    else:
        for rd in _repl_rows:
            sev = rd.get("severity", "info")
            st  = rd.get("status", "INFO")
            ws.cell(row=r, column=1, value=rd.get("check", ""))
            ws.cell(row=r, column=2, value=st)
            ws.cell(row=r, column=3, value=rd.get("details", ""))
            _style_row(ws, r, 4, sev)
            if st == "PASS":
                ws.cell(row=r, column=2).font = Font(bold=True, size=10, color="274E13")
            elif st == "WARN":
                ws.cell(row=r, column=2).font = Font(bold=True, size=10, color="B45F06")
            else:
                ws.cell(row=r, column=2).font = Font(bold=True, size=10, color="CC0000")
            r += 1


def write_critical_sets_sheet(wb, all_issues, found_info=None):
    ws = wb.create_sheet("Critical Sets")
    ws.sheet_view.showGridLines = False

    df = all_issues.filter(
        pl.col("issue_type").is_in(list(CRITICAL_ISSUE_TYPES))
    ).sort(["set_name", "severity", "Q Name"])
    _sect(ws, 1, "CRITICAL SETS  Structural validation issues", 8)
    next_row = _table(ws, 2, df, apply_view=False)

    if found_info:
        next_row += 1
        _sect(ws, next_row, "QUESTIONS FOUND IN CURRENT QUESTIONNAIRE", 3)
        next_row += 1
        _hdr(ws, next_row, ["Set", "Count found", "Questions"])
        next_row += 1
        for set_name, questions in found_info.items():
            ws.cell(row=next_row, column=1, value=set_name)
            ws.cell(row=next_row, column=2, value=len(questions))
            ws.cell(row=next_row, column=3, value=", ".join(questions) if questions else "none found")
            _style_row(ws, next_row, 3, "info" if questions else "high")
            next_row += 1
    ws.freeze_panes = "A1"
    _autofit(ws)


def write_questionnaire_structure_sheet(wb, all_issues):
    ws = wb.create_sheet("Questionnaire Structure")
    ws.sheet_view.showGridLines = False
    row = 1

    # Box 1: legacy relevant checks
    rel_df = all_issues.filter(
        pl.col("issue_type").is_in(list(RELEVANT_ISSUE_TYPES))
    ).sort(["severity", "Q Name"])
    _sect(ws, row, "RELEVANT CHANGES  KoBO skip-logic (relevant column)", 8)
    _rel_start = row + 1
    row = _table(ws, _rel_start, rel_df, apply_view=False)
    _apply_inline_diff_for_issue(
        ws,
        _rel_start,
        rel_df,
        [(s, d) for s, d in COL_MAP if s in rel_df.columns],
        {"relevant_modified"},
    )

    # Box 2: new placeholder / token / syntax checks
    row += 1
    struct_df = all_issues.filter(
        pl.col("issue_type").is_in(list(STRUCTURE_ISSUE_TYPES))
    ).sort(["severity", "Q Name", "field"])
    _sect(ws, row, "QUESTIONNAIRE STRUCTURE  Placeholders and KoBO references", 8)
    _table(ws, row + 1, struct_df, apply_view=False)

    # Avoid frozen panes in multi-box sheets (this was causing navigation/display issues).
    ws.freeze_panes = "A1"
    _autofit(ws)


def write_relevant_sheet(wb, all_issues):
    ws = wb.create_sheet("Relevant Changes")
    ws.sheet_view.showGridLines = False
    df = all_issues.filter(pl.col("issue_type").is_in(list(RELEVANT_ISSUE_TYPES)))
    _sect(ws, 1, "RELEVANT CHANGES  KoBO skip-logic (relevant column)", 8)
    _table(ws, 2, df)
    _autofit(ws)


def write_question_changes_sheet(wb, all_issues):
    ws = wb.create_sheet("Question Changes")
    ws.sheet_view.showGridLines = False
    df = all_issues.filter(pl.col("issue_type").is_in(list(QUESTION_CHANGE_TYPES)))
    col_map = [c for c in COL_MAP if c[0] != "set_name"]
    _sect(ws, 1, "QUESTION CHANGES  Presence, mandatory, label, type, hint, required, appearance, calculation, constraint, choices list", 8)
    _table(ws, 2, df, col_map=col_map)
    _apply_inline_diff_for_issue(ws, 2, df, [(s, d) for s, d in col_map if s in df.columns], {"label_mismatch", "round_parameter_change"})
    _autofit(ws)


def write_option_changes_sheet(wb, all_issues):
    ws = wb.create_sheet("Option Changes")
    ws.sheet_view.showGridLines = False
    df = all_issues.filter(pl.col("issue_type").is_in([
        "removed_option", "added_option", "option_label_mismatch",
    ]))
    col_map = [c for c in COL_MAP if c[0] != "set_name"]
    _sect(ws, 1, "OPTION CHANGES  Answer options for questions in both files", 8)
    _table(ws, 2, df, col_map=col_map)
    _apply_inline_diff_for_issue(ws, 2, df, [(s, d) for s, d in col_map if s in df.columns], {"option_label_mismatch"})
    _autofit(ws)


def export_report(all_issues, result_file, found_info=None,
                  rules=None, critical_issues=None, count_issues=None,
                  replacement_status=None):
    wb = Workbook()
    wb.remove(wb.active)
    write_summary_sheet(wb, all_issues, rules=rules,
                        critical_issues=critical_issues, count_issues=count_issues,
                        found_info=found_info, replacement_status=replacement_status)
    write_critical_sets_sheet(wb, all_issues, found_info=found_info)
    write_questionnaire_structure_sheet(wb, all_issues)
    write_question_changes_sheet(wb, all_issues)
    write_option_changes_sheet(wb, all_issues)
    wb.save(result_file)
    print(f"Report saved: {result_file}")
    print(f"Sheets: {wb.sheetnames}")


# -- run export ----------------------------------------------------------------
from pathlib import Path as _P
_q_path = _P(run["questionnaire_path"])
_report_file = str(_q_path.parent / (_q_path.stem + "_validation_report.xlsx"))
export_report(all_issues, _report_file, found_info=_found_info,
              rules=rules, critical_issues=critical_issues, count_issues=count_issues,
              replacement_status=globals().get("_replacement_status"))









Report saved: C:\Temp\DIEM_Monitoring questionnaire_kobo_en_AFG_20260415_test_validation_report.xlsx
Sheets: ['Summary', 'Critical Sets', 'Questionnaire Structure', 'Question Changes', 'Option Changes']


## Step 12 -- Validated questionnaire output

Produces `validated_questionnaire_kobo_<lang>_<iso3>_<date>.xlsx` in `output_dir`:

1. Copy country questionnaire
2. Inject **crop choices** (crop / crop2 / crop3) from the `Crop list` sheet, selected first
3. Inject **admin choices** (admin1 / admin2) fetched live from FAO AGOL -- no credentials needed
4. Restore template labels for `#placeholder#` questions (vanilla form before replacement)
5. Apply `#placeholder#` -> actual values from `Additional information` sheet throughout survey + choices

In [104]:
import urllib.request as _urlreq
import json as _json
from shutil import copy2 as _copy2
from datetime import date as _date

# Special answer codes appended to each crop list
_CROP_SPECIALS = {
    "crop" : [("777", "No crop production"), ("888", "Don't know"), ("999", "Refused")],
    "crop2": [("666", "No other crop"),       ("888", "Don't know"), ("999", "Refused")],
    "crop3": [("666", "No other crop"),       ("888", "Don't know"), ("999", "Refused")],
}


def _read_crop_choices(src_path: str, language: str) -> tuple[dict[str, list[tuple]], dict]:
    """
    Read 'Crop list' sheet -> {list_name: [(code, label), ...]} for crop/crop2/crop3.
    Also returns crop-selection diagnostics for reporting (expected exactly 10 selected).
    """
    status = {
        "expected_selected": 10,
        "selected_count": 0,
        "candidate_count": 0,
        "status": "FAIL",
        "severity": "high",
        "message": "Crop list sheet not found",
    }

    wb = openpyxl.load_workbook(src_path, data_only=True, read_only=True)
    if "Crop list" not in wb.sheetnames:
        wb.close()
        return {}, status
    rows = list(wb["Crop list"].iter_rows(values_only=True))
    wb.close()

    # Layout: row 1 = title, row 2 = empty/sub-title, row 3 = header  (skiprows=2)
    if len(rows) < 3:
        status["message"] = "Crop list sheet has no crop rows"
        return {}, status

    headers  = [str(h).strip() if h is not None else "" for h in rows[2]]
    sel_idx  = next((i for i, h in enumerate(headers) if "Select top" in h), None)
    code_idx = next((i for i, h in enumerate(headers) if "Dataset code" in h), None)
    if language == "fr":
        lbl_idx = next((i for i, h in enumerate(headers) if "Label" in h and "FR" in h), None)
    else:
        lbl_idx = next((i for i, h in enumerate(headers) if "Label" in h and "EN" in h), None)

    if code_idx is None or lbl_idx is None:
        status["message"] = "Crop list headers missing Dataset code / language label"
        return {}, status

    selected, unselected = [], []
    for row in rows[3:]:
        code = row[code_idx] if code_idx < len(row) else None
        lbl  = row[lbl_idx]  if lbl_idx  < len(row) else None
        sel  = row[sel_idx]  if sel_idx is not None and sel_idx < len(row) else None
        if code is None or str(code).strip() in ("", "nan"):
            continue
        entry = (str(code).strip(), str(lbl or "").strip())
        (selected if sel is not None and str(sel).strip() else unselected).append(entry)

    selected.sort(key=lambda x: x[0])
    unselected.sort(key=lambda x: x[0])
    combined = selected + unselected

    selected_count = len(selected)
    candidate_count = len(combined)
    status["selected_count"] = selected_count
    status["candidate_count"] = candidate_count

    if selected_count == 10:
        status["status"] = "PASS"
        status["severity"] = "pass"
        status["message"] = f"10 crops selected (OK) out of {candidate_count} candidates"
    elif selected_count == 0:
        status["status"] = "FAIL"
        status["severity"] = "high"
        status["message"] = f"No crops selected (0/10) out of {candidate_count} candidates"
    elif selected_count < 10:
        status["status"] = "WARN"
        status["severity"] = "medium"
        status["message"] = f"Only {selected_count}/10 crops selected out of {candidate_count} candidates"
    else:
        status["status"] = "WARN"
        status["severity"] = "medium"
        status["message"] = f"{selected_count}/10 crops selected (more than expected) out of {candidate_count} candidates"

    return ({name: combined + specials for name, specials in _CROP_SPECIALS.items()}, status)


def _fetch_admin_choices(iso3: str) -> dict[str, list[tuple]]:
    """
    Fetch adm1 and adm2 from public FAO AGOL REST service (no credentials needed).
    Returns:
      admin1: [(adm1_pcode, adm1_name), ...]
      admin2: [(adm2_pcode, adm2_name, adm1_pcode), ...]  <- adm1_pcode for choice_filter (col 5)
    """
    _BASE = ("https://services5.arcgis.com/sjP4Ugu5s0dZWLjd/arcgis/rest/services"
             "/Administrative_Boundaries_Reference_(view_layer)/FeatureServer")

    def _get(layer, fields):
        url = (f"{_BASE}/{layer}/query"
               f"?where=adm0_ISO3+%3D+%27{iso3}%27"
               f"&outFields={fields}&returnGeometry=false&outSR=4326&f=json")
        with _urlreq.urlopen(url, timeout=30) as r:
            return _json.loads(r.read().decode())["features"]

    adm1 = sorted(
        [(f["attributes"]["adm1_pcode"], f["attributes"]["adm1_name"])
         for f in _get(1, "adm1_name,adm1_pcode")],
        key=lambda x: x[0],
    )
    adm2 = sorted(
        [(f["attributes"]["adm2_pcode"], f["attributes"]["adm2_name"], f["attributes"]["adm1_pcode"])
         for f in _get(0, "adm2_name,adm2_pcode,adm1_pcode")],
        key=lambda x: x[0],
    )
    print(f"  AGOL  adm1: {len(adm1)} rows   adm2: {len(adm2)} rows")
    return {"admin1": adm1, "admin2": adm2}


def _rebuild_choices_sheet(wb, country_rows: dict[str, list[tuple]]) -> None:
    """
    Replace all rows belonging to country_rows list_names with fresh data.
    Non-country rows and empty separator rows are preserved in their original order.
    New country blocks are appended at the end, each preceded by an empty separator row.
    admin2 rows also populate col 5 (adm1_pcode) for choice_filter cascading.
    """
    ws       = wb["choices"]
    all_rows = list(ws.iter_rows(values_only=True))
    n_cols   = max((len(r) for r in all_rows), default=5)
    skip     = set(country_rows.keys())

    # Keep header + non-country rows + empty separator rows (r[0] is None)
    kept = [all_rows[0]] + [
        r for r in all_rows[1:]
        if r[0] is None or r[0] not in skip
    ]

    # Build new country blocks, each preceded by an empty separator row
    new_rows = []
    for list_name, entries in country_rows.items():
        new_rows.append([None] * n_cols)   # blank separator between blocks
        for entry in entries:
            row    = [None] * n_cols
            row[0] = list_name
            row[1] = entry[0]   # code / pcode
            row[2] = entry[1]   # label / name
            if len(entry) > 2 and n_cols >= 5:
                row[4] = entry[2]   # adm1_pcode in col 5 (choice_filter)
            new_rows.append(row)

    ws.delete_rows(1, ws.max_row)
    for r in kept + new_rows:
        ws.append(list(r))


def _apply_replacements_to_wb(wb, replacement_pairs: dict) -> None:
    """Apply #placeholder# -> value to every string cell in survey and choices sheets."""
    if not replacement_pairs:
        return
    for sheet_name in ("survey", "choices"):
        if sheet_name not in wb.sheetnames:
            continue
        for row in wb[sheet_name].iter_rows():
            for cell in row:
                if isinstance(cell.value, str) and "#" in cell.value:
                    v = cell.value
                    for key, val in replacement_pairs.items():
                        if key in v:
                            v = v.replace(key, str(val))
                    cell.value = v


def _count_choices_list_rows(wb, list_name: str) -> int:
    """Count rows in choices sheet belonging to a given list_name."""
    if "choices" not in wb.sheetnames:
        return 0
    ws = wb["choices"]
    return sum(
        1
        for r in ws.iter_rows(min_row=2, values_only=True)
        if str((r[0] if r and len(r) > 0 else "") or "").strip() == list_name
    )


def produce_validated_questionnaire(
    cfg             : dict,
    reference_survey: pl.DataFrame,
    replacement_pairs: dict,
) -> tuple[str, dict]:
    """
    Build validated_questionnaire_kobo_<lang>_<iso3>_<date>.xlsx:
      1. Copy country questionnaire to output_dir
      2. Inject crop choices from 'Crop list' sheet
      3. Inject admin choices from AGOL
      4. Restore #placeholder# labels from template (then replace in step 5)
      5. Apply all #placeholder# replacements from Additional information sheet
    Returns (saved_file_path, replacement_status).
    """
    src  = cfg["questionnaire_path"]
    iso3 = cfg["iso3"]
    lang = cfg["language"]
    out  = Path(cfg.get("output_dir") or str(Path(src).parent))
    dest = str(out / f"validated_questionnaire_kobo_{lang}_{iso3}_{_date.today():%Y%m%d}.xlsx")

    replacement_status = {"rows": []}

    def _add_row(check: str, status: str, details: str, severity: str):
        replacement_status["rows"].append({
            "check": check,
            "status": status,
            "details": details,
            "severity": severity,
        })

    print("Building validated questionnaire ...")
    print(f"  Source : {Path(src).name}")
    print(f"  Output : {dest}")

    _copy2(src, dest)
    wb = openpyxl.load_workbook(dest)

    # -- 2. Crop choices --------------------------------------------------------
    crop_rows, crop_sel_status = _read_crop_choices(src, lang)
    print(f"  Crop selection: {crop_sel_status['message']}")
    if crop_sel_status["status"] == "PASS":
        _add_row("Crop selection (top 10)", "PASS", crop_sel_status["message"], "pass")
    elif crop_sel_status["status"] == "WARN":
        _add_row("Crop selection (top 10)", "WARN", crop_sel_status["message"], "medium")
    else:
        _add_row("Crop selection (top 10)", "FAIL", crop_sel_status["message"], "high")

    if crop_rows:
        print(f"  Crops  : {sum(len(v) for v in crop_rows.values())} entries  "
              f"({', '.join(crop_rows)})")

    # -- 3. Admin choices from AGOL ---------------------------------------------
    admin_rows: dict = {}
    agol_error = None
    try:
        admin_rows = _fetch_admin_choices(iso3)
    except Exception as e:
        agol_error = str(e)
        print(f"  Warning: AGOL fetch failed -- admin choices not updated  ({e})")

    adm1_n = len(admin_rows.get("admin1", []))
    adm2_n = len(admin_rows.get("admin2", []))
    if agol_error:
        _add_row("AGOL fetch", "FAIL", f"Fetch failed: {agol_error}", "high")
    elif adm1_n > 0 and adm2_n > 0:
        _add_row("AGOL fetch", "PASS", f"admin1={adm1_n}, admin2={adm2_n}", "pass")
    else:
        _add_row("AGOL fetch", "WARN", f"admin1={adm1_n}, admin2={adm2_n}", "medium")

    country_rows = {**crop_rows, **admin_rows}
    if country_rows:
        _rebuild_choices_sheet(wb, country_rows)

    # Verify choices replacement counts in workbook
    for list_name in ["crop", "crop2", "crop3"]:
        expected = len(crop_rows.get(list_name, []))
        actual = _count_choices_list_rows(wb, list_name)
        if expected > 0 and actual == expected:
            _add_row(f"Choices replaced: {list_name}", "PASS", f"{actual} rows written", "pass")
        elif expected == 0:
            _add_row(f"Choices replaced: {list_name}", "FAIL", "No rows prepared from Crop list", "high")
        else:
            _add_row(f"Choices replaced: {list_name}", "FAIL", f"Expected {expected}, wrote {actual}", "high")

    for list_name in ["admin1", "admin2"]:
        expected = len(admin_rows.get(list_name, []))
        actual = _count_choices_list_rows(wb, list_name)
        if expected > 0 and actual == expected:
            _add_row(f"Choices replaced: {list_name}", "PASS", f"{actual} rows written", "pass")
        elif agol_error:
            _add_row(f"Choices replaced: {list_name}", "FAIL", "Skipped because AGOL fetch failed", "high")
        elif expected == 0:
            _add_row(f"Choices replaced: {list_name}", "WARN", "No rows fetched from AGOL", "medium")
        else:
            _add_row(f"Choices replaced: {list_name}", "FAIL", f"Expected {expected}, wrote {actual}", "high")

    # -- 4. Restore template labels for #placeholder# questions -----------------
    #      Reset labels to canonical template form so step 5 substitutes correctly.
    if "survey" in wb.sheetnames and reference_survey.height > 0:
        ws_s      = wb["survey"]
        hdr       = [str(c.value or "").strip() for c in next(ws_s.iter_rows(min_row=1, max_row=1))]
        name_idx  = next((i for i, h in enumerate(hdr) if h == "name"), None)
        label_col = _find_label_col(hdr, lang)
        label_idx = hdr.index(label_col) if label_col and label_col in hdr else None

        if name_idx is not None and label_idx is not None:
            ref_ph = {
                r["Q Name"]: r["label"]
                for r in reference_survey
                    .filter(pl.col("label").str.contains(r"#[^#]+#"))
                    .select(["Q Name", "label"])
                    .iter_rows(named=True)
            }
            for row in ws_s.iter_rows(min_row=2):
                qn = str(row[name_idx].value or "").strip()
                if qn in ref_ph:
                    row[label_idx].value = ref_ph[qn]

    # -- 5. Apply #placeholder# -> actual values ---------------------------------
    _apply_replacements_to_wb(wb, replacement_pairs)

    wb.save(dest)
    print(f"  Saved  : {dest}")

    print("  Replacement checks:")
    for rr in replacement_status["rows"]:
        print(f"    - {rr['check']}: {rr['status']}  ({rr['details']})")

    return dest, replacement_status


# -- Run ------------------------------------------------------------------------
_validated_path, _replacement_status = produce_validated_questionnaire(cfg, reference_survey, replacement_pairs)

# Refresh report so Summary includes replacement status from Step 12
if all(k in globals() for k in ["export_report", "all_issues", "_report_file", "_found_info", "rules", "critical_issues", "count_issues"]):
    export_report(
        all_issues,
        _report_file,
        found_info=_found_info,
        rules=rules,
        critical_issues=critical_issues,
        count_issues=count_issues,
        replacement_status=_replacement_status,
    )
    print(f"  Report refreshed with replacement status: {_report_file}")
else:
    print("  Note: Replacement status prepared, but report context is not loaded in memory.")



Building validated questionnaire ...
  Source : DIEM_Monitoring questionnaire_kobo_en_AFG_20260415_test.xlsx
  Output : C:\Temp\validated_questionnaire_kobo_en_AFG_20260422.xlsx
  Crop selection: Only 8/10 crops selected out of 90 candidates
  Crops  : 279 entries  (crop, crop2, crop3)
  AGOL  adm1: 34 rows   adm2: 399 rows
  Saved  : C:\Temp\validated_questionnaire_kobo_en_AFG_20260422.xlsx
  Replacement checks:
    - Crop selection (top 10): WARN  (Only 8/10 crops selected out of 90 candidates)
    - AGOL fetch: PASS  (admin1=34, admin2=399)
    - Choices replaced: crop: PASS  (93 rows written)
    - Choices replaced: crop2: PASS  (93 rows written)
    - Choices replaced: crop3: PASS  (93 rows written)
    - Choices replaced: admin1: PASS  (34 rows written)
    - Choices replaced: admin2: PASS  (399 rows written)
Report saved: C:\Temp\DIEM_Monitoring questionnaire_kobo_en_AFG_20260415_test_validation_report.xlsx
Sheets: ['Summary', 'Critical Sets', 'Questionnaire Structure', 'Questio